In [1]:
# 셀 1: 라이브러리 import 및 환경 설정
import requests
import json
import pandas as pd
from pprint import pprint

# Colab 환경 확인
try:
    import google.colab
    IS_COLAB = True
    print("Colab 환경 감지")
except ImportError:
    IS_COLAB = False
    print("로컬 환경")

print(f"\nrequests 버전: {requests.__version__}")
print(f"pandas 버전: {pd.__version__}")

로컬 환경

requests 버전: 2.34.2
pandas 버전: 3.0.3


In [2]:
# 셀 2 (로컬용): V-World 인증키 보안 로드
# Colab의 userdata.get() 대신 .env 파일에서 로드
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일을 읽어 환경변수에 로드

VWORLD_KEY = os.getenv('VWORLD_KEY')

if VWORLD_KEY:
    print(f"인증키 로드 완료 (마스킹): {VWORLD_KEY[:4]}...{VWORLD_KEY[-4:]}")
    print(f"인증키 길이: {len(VWORLD_KEY)}자")
else:
    print("오류: .env 파일에서 VWORLD_KEY를 찾을 수 없다.")
    print(f"현재 작업 디렉토리: {os.getcwd()}")
    print(".env 파일이 노트북과 같은 폴더에 있는지 확인할 것.")

인증키 로드 완료 (마스킹): 8C07...3D4E
인증키 길이: 36자


In [3]:
# 셀 3 수정: V-World 단일 좌표 조회 테스트 (HTTPS + 진단)
# 미추홀구청 좌표 기준 용도지역 조회

# 테스트 좌표 — 미추홀구청 (인천시 미추홀구)
TEST_LAT = 37.4633
TEST_LON = 126.6502
LOCATION_NAME = "미추홀구청"

# V-World 데이터 API 호출 — HTTPS 사용
url = "https://api.vworld.kr/req/data"   # http → https 변경
params = {
    "service": "data",
    "request": "GetFeature",
    "data": "LT_C_UD801",                              # 용도지역도
    "key": VWORLD_KEY,
    "domain": "localhost",
    "geomFilter": f"POINT({TEST_LON} {TEST_LAT})",     # V-World는 lon, lat 순서
    "format": "json",
    "size": "10",
    "page": "1"
}

print(f"[조회 위치] {LOCATION_NAME}")
print(f"[좌표] lat={TEST_LAT}, lon={TEST_LON}")
print(f"[조회 데이터] LT_C_UD801 (용도지역도)")
print(f"[엔드포인트] {url}\n")

try:
    response = requests.get(url, params=params, timeout=15)

    # 진단 정보 출력
    print(f"[요청 URL] {response.url}")  # 실제 호출된 URL 확인
    print(f"[HTTP 상태] {response.status_code}")
    print(f"[Content-Type] {response.headers.get('Content-Type', 'N/A')}")
    print(f"[응답 길이] {len(response.text):,}자\n")

    if response.status_code == 200:
        # JSON 파싱 시도
        try:
            result = response.json()
            print("[응답 최상위 키]")
            pprint(list(result.keys()))

            # V-World 응답 status 확인
            if 'response' in result:
                vworld_status = result['response'].get('status', 'UNKNOWN')
                print(f"\n[V-World 응답 상태] {vworld_status}")

                if vworld_status == 'OK':
                    print("\n[응답 전체 구조 미리보기 (앞 2500자)]")
                    print(json.dumps(result, ensure_ascii=False, indent=2)[:2500])
                elif vworld_status == 'NOT_FOUND':
                    print("\n해당 좌표에 데이터 없음. 다른 좌표 시도 필요.")
                else:
                    print(f"\n[V-World 오류 응답]")
                    print(json.dumps(result, ensure_ascii=False, indent=2))
        except json.JSONDecodeError:
            print(f"[JSON 파싱 실패] 응답이 JSON 형식 아님")
            print(f"[응답 본문 앞 500자]\n{response.text[:500]}")
    else:
        print(f"[오류 본문]\n{response.text}")

except requests.exceptions.Timeout:
    print("[타임아웃] V-World 서버 응답 지연. 잠시 후 재시도 권고.")
except requests.exceptions.RequestException as e:
    print(f"[요청 예외] {e}")

[조회 위치] 미추홀구청
[좌표] lat=37.4633, lon=126.6502
[조회 데이터] LT_C_UD801 (용도지역도)
[엔드포인트] https://api.vworld.kr/req/data

[요청 URL] https://api.vworld.kr/req/data?service=data&request=GetFeature&data=LT_C_UD801&key=8C074AE5-36AA-3731-9D79-02CB19043D4E&domain=localhost&geomFilter=POINT%28126.6502+37.4633%29&format=json&size=10&page=1
[HTTP 상태] 200
[Content-Type] application/json;charset=UTF-8
[응답 길이] 239자

[응답 최상위 키]
['response']

[V-World 응답 상태] NOT_FOUND

해당 좌표에 데이터 없음. 다른 좌표 시도 필요.


In [4]:
# 셀 4: 데이터셋 ID 진단 (단계적 검증)
# 목적: NOT_FOUND가 데이터셋 ID 문제인지, 좌표/필터 문제인지 분리 진단

TEST_LAT = 37.4633
TEST_LON = 126.6502
url = "https://api.vworld.kr/req/data"

def test_dataset(data_id, label, use_geom=True):
    """단일 데이터셋 호출 후 응답 status만 간결히 출력"""
    params = {
        "service": "data",
        "request": "GetFeature",
        "data": data_id,
        "key": VWORLD_KEY,
        "domain": "localhost",
        "format": "json",
        "size": "5",
        "page": "1",
    }
    if use_geom:
        params["geomFilter"] = f"POINT({TEST_LON} {TEST_LAT})"
        params["crs"] = "EPSG:4326"          # 좌표계 명시 추가
    
    try:
        r = requests.get(url, params=params, timeout=15)
        if r.status_code != 200:
            print(f"  [{label:25s}] HTTP {r.status_code}")
            return None
        result = r.json()
        status = result.get('response', {}).get('status', 'UNKNOWN')
        feature_count = 0
        if status == 'OK':
            features = result['response']['result']['featureCollection']['features']
            feature_count = len(features)
        print(f"  [{label:25s}] status={status}, features={feature_count}")
        return result if status == 'OK' else None
    except Exception as e:
        print(f"  [{label:25s}] 예외: {type(e).__name__}")
        return None

print("=" * 60)
print("[진단 1] 기존 ID (LT_C_UD801) — 데이터 API에 없을 것으로 예상")
print("=" * 60)
test_dataset("LT_C_UD801", "용도지역 통합(WMS용)", use_geom=True)

print("\n" + "=" * 60)
print("[진단 2] 정상 작동 확인용 — 시도 행정구역")
print("=" * 60)
test_dataset("LT_C_ADSIDO_INFO", "시도 행정구역", use_geom=False)  
# geom 없이 전체 조회 (size=5로 제한)

print("\n" + "=" * 60)
print("[진단 3] 미추홀구청 좌표에서 4개 용도지역 개별 조회")
print("=" * 60)
results = {}
for data_id, label in [
    ("LT_C_UQ111", "도시지역"),
    ("LT_C_UQ112", "관리지역"),
    ("LT_C_UQ113", "농림지역"),
    ("LT_C_UQ114", "자연환경보전지역"),
]:
    res = test_dataset(data_id, label, use_geom=True)
    if res:
        results[data_id] = res

# 정상 응답 1개를 자세히 출력 (properties 구조 확인용)
print("\n" + "=" * 60)
print("[정상 응답 properties 구조 — UQA 코드 위치 확인용]")
print("=" * 60)
if results:
    first_id = next(iter(results))
    first_feature = results[first_id]['response']['result']['featureCollection']['features'][0]
    print(f"\n사용된 데이터셋: {first_id}")
    print(f"\nproperties 필드 전체:")
    pprint(first_feature['properties'])
    print(f"\ngeometry.type: {first_feature['geometry']['type']}")
    print(f"geometry 좌표 샘플 (첫 3개): {str(first_feature['geometry']['coordinates'])[:200]}")
else:
    print("정상 응답 없음. 좌표를 다른 곳으로 변경하여 재시도 필요.")

[진단 1] 기존 ID (LT_C_UD801) — 데이터 API에 없을 것으로 예상
  [용도지역 통합(WMS용)            ] status=NOT_FOUND, features=0

[진단 2] 정상 작동 확인용 — 시도 행정구역
  [시도 행정구역                  ] 예외: JSONDecodeError

[진단 3] 미추홀구청 좌표에서 4개 용도지역 개별 조회
  [도시지역                     ] status=OK, features=1
  [관리지역                     ] status=NOT_FOUND, features=0
  [농림지역                     ] status=NOT_FOUND, features=0
  [자연환경보전지역                 ] status=NOT_FOUND, features=0

[정상 응답 properties 구조 — UQA 코드 위치 확인용]

사용된 데이터셋: LT_C_UQ111

properties 필드 전체:
{'dnum': '0125',
 'dyear': '2025',
 'sido_name': '인천광역시',
 'sigg_name': '미추홀구',
 'uname': '제2종일반주거지역'}

geometry.type: MultiPolygon
geometry 좌표 샘플 (첫 3개): [[[[126.64977613610746, 37.45829800852041], [126.64975741885087, 37.458334263137786], [126.64953967986047, 37.458786728059245], [126.64961019516736, 37.45883532177466], [126.64969515506009, 37.4588863


### 인천 9개 군구 폴리곤 일괄 수집

In [6]:
# 셀 5: 단계 3 준비 — 회피 대상 및 대상 시군구 정의

# 회피 대상 용도지역 (uname 한글명 기반 매핑)
# UQA300번대(공업지역)는 모두 도시지역(LT_C_UQ111)의 하위 분류이므로 단일 데이터셋 호출로 충분
AVOID_UNAMES = {
    "전용공업지역": "UQA310",
    "일반공업지역": "UQA320",
    "준공업지역":   "UQA330",
}

# 인천광역시 9개 군구 (옹진군 제외 — 도서지역, 본 알고리즘 OD 시나리오 외)
INCHEON_SIDO = "인천광역시"
INCHEON_DISTRICTS = [
    "중구", "동구", "미추홀구", "연수구", "남동구",
    "부평구", "계양구", "서구", "강화군",
]

print(f"회피 대상 용도지역 ({len(AVOID_UNAMES)}종):")
for uname, code in AVOID_UNAMES.items():
    print(f"  - {code}: {uname}")

print(f"\n대상 시군구 ({len(INCHEON_DISTRICTS)}개):")
print(f"  {INCHEON_SIDO} {', '.join(INCHEON_DISTRICTS)}")

회피 대상 용도지역 (3종):
  - UQA310: 전용공업지역
  - UQA320: 일반공업지역
  - UQA330: 준공업지역

대상 시군구 (9개):
  인천광역시 중구, 동구, 미추홀구, 연수구, 남동구, 부평구, 계양구, 서구, 강화군


In [ ]:
# [폐기] 사유: attrFilter 다중조건 미지원
# # 셀 6: 단일 군구의 도시지역 폴리곤을 페이지네이션으로 일괄 수집하는 함수
# import time

# def fetch_district_features(sido_name, sigg_name, vworld_key, page_size=1000, max_pages=50):
#     """
#     특정 시도+군구 조합의 LT_C_UQ111(도시지역) 모든 features를 페이지네이션 수집.
    
#     Args:
#         sido_name: 시도명 (예: "인천광역시")
#         sigg_name: 시군구명 (예: "미추홀구")
#         vworld_key: V-World API 인증키
#         page_size: 페이지당 최대 features 수 (V-World 상한 1000)
#         max_pages: 안전장치 — 최대 페이지 수
    
#     Returns:
#         dict: {
#             'features': [...],     # 전체 features 리스트
#             'total': int,          # V-World가 보고한 전체 개수
#             'fetched': int,        # 실제 수집된 개수
#             'pages_called': int,   # 호출한 페이지 수
#             'status': str,         # 'OK' | 'NOT_FOUND' | 'ERROR'
#         }
#     """
#     url = "https://api.vworld.kr/req/data"
#     all_features = []
#     total_reported = 0
#     page = 1
    
#     while page <= max_pages:
#         params = {
#             "service": "data",
#             "request": "GetFeature",
#             "data": "LT_C_UQ111",
#             "key": vworld_key,
#             "domain": "localhost",
#             "format": "json",
#             "size": str(page_size),
#             "page": str(page),
#             "attrFilter": f"sido_name:=:{sido_name}|sigg_name:=:{sigg_name}",
#             "crs": "EPSG:4326",
#         }
        
#         try:
#             r = requests.get(url, params=params, timeout=30)
#         except requests.exceptions.RequestException as e:
#             return {'features': all_features, 'total': total_reported,
#                     'fetched': len(all_features), 'pages_called': page - 1,
#                     'status': f'ERROR: {type(e).__name__}'}
        
#         if r.status_code != 200:
#             return {'features': all_features, 'total': total_reported,
#                     'fetched': len(all_features), 'pages_called': page - 1,
#                     'status': f'ERROR: HTTP {r.status_code}'}
        
#         try:
#             result = r.json()
#         except json.JSONDecodeError:
#             return {'features': all_features, 'total': total_reported,
#                     'fetched': len(all_features), 'pages_called': page - 1,
#                     'status': 'ERROR: JSONDecodeError'}
        
#         vworld_status = result.get('response', {}).get('status', 'UNKNOWN')
        
#         if vworld_status == 'NOT_FOUND':
#             return {'features': all_features, 'total': 0,
#                     'fetched': 0, 'pages_called': page,
#                     'status': 'NOT_FOUND'}
        
#         if vworld_status != 'OK':
#             return {'features': all_features, 'total': total_reported,
#                     'fetched': len(all_features), 'pages_called': page,
#                     'status': f'ERROR: V-World status={vworld_status}'}
        
#         # 정상 응답 — features 누적
#         features = result['response']['result']['featureCollection']['features']
#         all_features.extend(features)
        
#         # 페이지 메타 정보 추출 (V-World 응답 구조: response.record)
#         record = result['response'].get('record', {})
#         try:
#             total_reported = int(record.get('total', 0))
#         except (ValueError, TypeError):
#             total_reported = len(all_features)
        
#         # 종료 조건: 전체 수집 완료
#         if len(all_features) >= total_reported:
#             break
        
#         # 다음 페이지 진입 전 V-World 부하 완화 — 짧은 대기
#         time.sleep(0.3)
#         page += 1
    
#     return {
#         'features': all_features,
#         'total': total_reported,
#         'fetched': len(all_features),
#         'pages_called': page,
#         'status': 'OK',
#     }

In [7]:
# 셀 6 (재작성): LT_C_ADSIGG_INFO에서 시군구 폴리곤 가져오기
# 단일검색 가능 필드: sig_kor_nm

def fetch_sigungu_polygon(sigg_name, sido_name, vworld_key):
    """
    특정 시군구의 행정구역 폴리곤을 가져온다.
    
    Args:
        sigg_name: 시군구명 (예: "미추홀구")
        sido_name: 시도명 (예: "인천광역시") — 명칭 충돌 검증용
        vworld_key: V-World 인증키
    
    Returns:
        dict: {
            'feature': dict | None,    # 매칭된 단일 feature
            'bbox': tuple | None,      # (min_lon, min_lat, max_lon, max_lat)
            'status': str,
        }
    """
    url = "https://api.vworld.kr/req/data"
    params = {
        "service": "data",
        "request": "GetFeature",
        "data": "LT_C_ADSIGG_INFO",
        "key": vworld_key,
        "domain": "localhost",
        "format": "json",
        "size": "10",
        "page": "1",
        "attrFilter": f"sig_kor_nm:=:{sigg_name}",
        "crs": "EPSG:4326",
    }
    
    try:
        r = requests.get(url, params=params, timeout=15)
    except requests.exceptions.RequestException as e:
        return {'feature': None, 'bbox': None, 'status': f'ERROR: {type(e).__name__}'}
    
    if r.status_code != 200:
        return {'feature': None, 'bbox': None, 'status': f'ERROR: HTTP {r.status_code}'}
    
    try:
        result = r.json()
    except json.JSONDecodeError:
        return {'feature': None, 'bbox': None, 'status': 'ERROR: JSONDecodeError'}
    
    vworld_status = result.get('response', {}).get('status', 'UNKNOWN')
    if vworld_status != 'OK':
        return {'feature': None, 'bbox': None, 'status': vworld_status}
    
    features = result['response']['result']['featureCollection']['features']
    
    # 시도명으로 명칭 충돌 해결 (예: "중구"는 6개 시도에 존재)
    # LT_C_ADSIGG_INFO의 properties에 full_nm이 "인천광역시 미추홀구" 형태로 들어있을 것
    matched = None
    for f in features:
        full_nm = f['properties'].get('full_nm', '')
        if sido_name in full_nm and sigg_name in full_nm:
            matched = f
            break
    
    if matched is None:
        return {'feature': None, 'bbox': None, 'status': f'NOT_FOUND_IN_SIDO ({len(features)}개 후보 중 매칭 실패)'}
    
    # BBOX 계산 — geometry 좌표를 재귀 순회하여 min/max 추출
    def extract_bbox(coords):
        """중첩 리스트에서 모든 (lon, lat) 좌표 추출"""
        if isinstance(coords[0], (int, float)):
            return [(coords[0], coords[1])]
        result = []
        for c in coords:
            result.extend(extract_bbox(c))
        return result
    
    all_points = extract_bbox(matched['geometry']['coordinates'])
    lons = [p[0] for p in all_points]
    lats = [p[1] for p in all_points]
    bbox = (min(lons), min(lats), max(lons), max(lats))
    
    return {'feature': matched, 'bbox': bbox, 'status': 'OK'}

In [8]:
# 셀 7-1: attrFilter syntax 진단
# 4가지 변형을 호출하여 응답 본문을 직접 비교한다.

url = "https://api.vworld.kr/req/data"

base_params = {
    "service": "data",
    "request": "GetFeature",
    "data": "LT_C_UQ111",
    "key": VWORLD_KEY,
    "domain": "localhost",
    "format": "json",
    "size": "5",
    "page": "1",
    "crs": "EPSG:4326",
}

variants = [
    ("A. 단일 :=: sigg만",       {"attrFilter": "sigg_name:=:미추홀구"}),
    ("B. 단일 :like: sigg만",    {"attrFilter": "sigg_name:like:미추홀구"}),
    ("C. 다중 :=: 파이프 결합",  {"attrFilter": "sido_name:=:인천광역시|sigg_name:=:미추홀구"}),
    ("D. 다중 :like: 파이프",    {"attrFilter": "sido_name:like:인천광역시|sigg_name:like:미추홀구"}),
]

for label, extra in variants:
    params = {**base_params, **extra}
    print("=" * 70)
    print(f"[{label}]")
    print(f"  attrFilter: {extra['attrFilter']}")
    
    try:
        r = requests.get(url, params=params, timeout=15)
    except Exception as e:
        print(f"  [예외] {type(e).__name__}: {e}")
        continue
    
    print(f"  [요청 URL] {r.url}")           # 실제 호출된 URL — 인코딩 확인용
    print(f"  [HTTP 상태] {r.status_code}")
    print(f"  [Content-Type] {r.headers.get('Content-Type', 'N/A')}")
    print(f"  [응답 길이] {len(r.text):,}자")
    
    # 응답 본문 일부 출력 (앞 400자)
    print(f"  [응답 본문 앞부분]")
    print(f"  {r.text[:400]}")
    
    # JSON 파싱 시도
    try:
        result = r.json()
        status = result.get('response', {}).get('status', 'UNKNOWN')
        if status == 'OK':
            features = result['response']['result']['featureCollection']['features']
            print(f"  [결과] status=OK, features={len(features)}개")
        else:
            print(f"  [결과] status={status}")
    except json.JSONDecodeError:
        print(f"  [결과] JSONDecodeError — 응답이 JSON 형식 아님")
    
    print()

[A. 단일 :=: sigg만]
  attrFilter: sigg_name:=:미추홀구
  [요청 URL] https://api.vworld.kr/req/data?service=data&request=GetFeature&data=LT_C_UQ111&key=8C074AE5-36AA-3731-9D79-02CB19043D4E&domain=localhost&format=json&size=5&page=1&crs=EPSG%3A4326&attrFilter=sigg_name%3A%3D%3A%EB%AF%B8%EC%B6%94%ED%99%80%EA%B5%AC
  [HTTP 상태] 200
  [Content-Type] application/json;charset=UTF-8
  [응답 길이] 325자
  [응답 본문 앞부분]
  {"response" : {"service" : {"name" : "data", "version" : "2.0", "operation" : "GetFeature", "time" : "14(ms)"}, "status" : "ERROR", "error" : {"level" : "1", "code" : "INVALID_RANGE", "text" : "요청 파라미터의 값이 유효한 범위를 넘었습니다. 유효한 파라미터 값의 범위 : 다음 2가지 [geomFilter, attrFilter(단일검색="Y"인 속성명 포함)] 중 하나 이상을 포함해야 합니다., 입력한 파라미터 값 : "}}}
  [결과] JSONDecodeError — 응답이 JSON 형식 아님

[B. 단일 :like: sigg만]
  attrFilter: sigg_name:like:미추홀구
  [요청 URL] https://api.vworld.kr/req/data?service=data&request=GetFeature&data=LT_C_UQ111&key=8C074AE5-36AA-3731-9D79-02CB19043D4E&domain=localhost&format=json&size=5&page=1&crs=

In [9]:
# 셀 7 (재작성): 시군구 BBOX 기반 LT_C_UQ111 features 수집

def fetch_urban_features_by_bbox(bbox, target_sigg_name, vworld_key,
                                  page_size=1000, max_pages=50):
    """
    BBOX 영역의 LT_C_UQ111 features를 페이지네이션 수집 후
    target_sigg_name과 정확히 매칭되는 것만 반환.
    
    Args:
        bbox: (min_lon, min_lat, max_lon, max_lat)
        target_sigg_name: 후처리 필터링용 시군구명
        vworld_key, page_size, max_pages: 동일
    
    Returns:
        dict: {
            'features': list,        # 시군구 매칭 features
            'total_in_bbox': int,    # BBOX 내 전체 (다른 시군구 포함)
            'matched': int,
            'status': str,
        }
    """
    url = "https://api.vworld.kr/req/data"
    min_lon, min_lat, max_lon, max_lat = bbox
    
    # V-World geomFilter syntax: BOX(min_lon, min_lat, max_lon, max_lat)
    geom_filter = f"BOX({min_lon},{min_lat},{max_lon},{max_lat})"
    
    all_features = []
    page = 1
    
    while page <= max_pages:
        params = {
            "service": "data",
            "request": "GetFeature",
            "data": "LT_C_UQ111",
            "key": vworld_key,
            "domain": "localhost",
            "format": "json",
            "size": str(page_size),
            "page": str(page),
            "geomFilter": geom_filter,
            "crs": "EPSG:4326",
        }
        
        try:
            r = requests.get(url, params=params, timeout=30)
        except requests.exceptions.RequestException as e:
            return {'features': [], 'total_in_bbox': len(all_features),
                    'matched': 0, 'status': f'ERROR: {type(e).__name__}'}
        
        if r.status_code != 200:
            return {'features': [], 'total_in_bbox': len(all_features),
                    'matched': 0, 'status': f'ERROR: HTTP {r.status_code}'}
        
        try:
            result = r.json()
        except json.JSONDecodeError:
            # 응답 본문 일부를 status에 포함하여 디버깅 용이하게
            return {'features': [], 'total_in_bbox': len(all_features),
                    'matched': 0, 'status': f'ERROR: JSONDecodeError. 응답: {r.text[:200]}'}
        
        vworld_status = result.get('response', {}).get('status', 'UNKNOWN')
        if vworld_status == 'NOT_FOUND':
            break
        if vworld_status != 'OK':
            return {'features': [], 'total_in_bbox': len(all_features),
                    'matched': 0, 'status': f'ERROR: V-World status={vworld_status}'}
        
        features = result['response']['result']['featureCollection']['features']
        all_features.extend(features)
        
        record = result['response'].get('record', {})
        try:
            total = int(record.get('total', 0))
        except (ValueError, TypeError):
            total = len(all_features)
        
        if len(all_features) >= total:
            break
        
        time.sleep(0.3)
        page += 1
    
    # 후처리: target_sigg_name 정확히 매칭되는 것만 필터링
    matched_features = [
        f for f in all_features
        if f['properties'].get('sigg_name') == target_sigg_name
    ]
    
    return {
        'features': matched_features,
        'total_in_bbox': len(all_features),
        'matched': len(matched_features),
        'status': 'OK',
    }

In [10]:
# 셀 8: 미추홀구로 단독 검증 — 2단계 파이프라인 통합 테스트

print(f"=" * 60)
print(f"[1단계] 시군구 행정구역 폴리곤 조회")
print(f"=" * 60)

sigungu_result = fetch_sigungu_polygon("미추홀구", INCHEON_SIDO, VWORLD_KEY)
print(f"  status: {sigungu_result['status']}")

if sigungu_result['status'] != 'OK':
    print("  → 1단계 실패. 진행 중단.")
else:
    bbox = sigungu_result['bbox']
    full_nm = sigungu_result['feature']['properties'].get('full_nm', 'N/A')
    print(f"  매칭: {full_nm}")
    print(f"  BBOX: ({bbox[0]:.5f}, {bbox[1]:.5f}, {bbox[2]:.5f}, {bbox[3]:.5f})")
    print(f"  BBOX 범위: 경도 {bbox[2]-bbox[0]:.4f}, 위도 {bbox[3]-bbox[1]:.4f}")
    
    print(f"\n{'=' * 60}")
    print(f"[2단계] BBOX 기반 도시지역 features 수집")
    print(f"{'=' * 60}")
    
    urban_result = fetch_urban_features_by_bbox(
        bbox, "미추홀구", VWORLD_KEY
    )
    print(f"  status: {urban_result['status']}")
    print(f"  BBOX 내 전체 features: {urban_result['total_in_bbox']:,}개")
    print(f"  미추홀구 정확 매칭: {urban_result['matched']:,}개")
    
    if urban_result['features']:
        # 회피 대상 추출
        from collections import Counter
        all_unames = Counter(f['properties'].get('uname', 'N/A') for f in urban_result['features'])
        avoid_features = [
            f for f in urban_result['features']
            if f['properties'].get('uname') in AVOID_UNAMES
        ]
        avoid_unames = Counter(f['properties']['uname'] for f in avoid_features)
        
        print(f"\n[전체 uname 분포 (미추홀구)]")
        for uname, count in all_unames.most_common():
            marker = " ← 회피" if uname in AVOID_UNAMES else ""
            print(f"  {uname}: {count}개{marker}")
        
        print(f"\n[회피 대상 추출 결과]")
        print(f"  회피 대상 매칭 총: {len(avoid_features)}개")
        for uname, count in avoid_unames.most_common():
            code = AVOID_UNAMES.get(uname, "?")
            print(f"  [{code}] {uname}: {count}개")

[1단계] 시군구 행정구역 폴리곤 조회
  status: OK
  매칭: 인천광역시 미추홀구
  BBOX: (126.62901, 37.42999, 126.70294, 37.48403)
  BBOX 범위: 경도 0.0739, 위도 0.0540

[2단계] BBOX 기반 도시지역 features 수집
  status: OK
  BBOX 내 전체 features: 305개
  미추홀구 정확 매칭: 171개

[전체 uname 분포 (미추홀구)]
  제3종일반주거지역: 47개
  준주거지역: 31개
  제2종일반주거지역: 26개
  자연녹지지역: 23개
  제1종일반주거지역: 15개
  일반상업지역: 11개
  준공업지역: 11개 ← 회피
  일반공업지역: 4개 ← 회피
  근린상업지역: 1개
  보전녹지지역: 1개
  중심상업지역: 1개

[회피 대상 추출 결과]
  회피 대상 매칭 총: 15개
  [UQA330] 준공업지역: 11개
  [UQA320] 일반공업지역: 4개


#### 9개 군구 일괄 수집

In [11]:
# 셀 9: 인천 9개 군구 일괄 수집
# 셀 6, 7의 함수를 재사용. 시군구별로 1단계+2단계 순차 호출.
import time

start_time = time.time()

# 결과 저장용 자료구조
all_avoid_features = []        # 9개 군구의 회피 대상 features 통합
sigungu_summary = []           # 군구별 요약 통계
sigungu_polygons = {}          # 군구별 행정구역 폴리곤 (시각화 단계에서 활용 예정)
all_features_by_sigungu = {}   # 군구별 도시지역 전체 features (디버깅용)

print(f"[수집 시작] 인천광역시 9개 군구\n")
print(f"{'시군구':<8} | {'1단계':<6} | {'2단계':<6} | {'전체':>6} | {'매칭':>6} | {'회피':>5} | {'소요(s)':>7}")
print("-" * 70)

for idx, sigg_name in enumerate(INCHEON_DISTRICTS, start=1):
    t0 = time.time()
    
    # 1단계: 시군구 행정구역 폴리곤
    sigungu_result = fetch_sigungu_polygon(sigg_name, INCHEON_SIDO, VWORLD_KEY)
    
    if sigungu_result['status'] != 'OK':
        elapsed = time.time() - t0
        print(f"{sigg_name:<8} | {sigungu_result['status']:<6} | {'-':<6} | "
              f"{'-':>6} | {'-':>6} | {'-':>5} | {elapsed:>7.1f}")
        sigungu_summary.append({
            'sigg_name': sigg_name, 'status_1': sigungu_result['status'],
            'status_2': None, 'total_in_bbox': 0,
            'matched': 0, 'avoid': 0, 'elapsed': elapsed,
        })
        continue
    
    sigungu_polygons[sigg_name] = sigungu_result['feature']
    bbox = sigungu_result['bbox']
    
    # 2단계: BBOX 기반 도시지역 features 수집
    urban_result = fetch_urban_features_by_bbox(bbox, sigg_name, VWORLD_KEY)
    elapsed = time.time() - t0
    
    if urban_result['status'] != 'OK':
        print(f"{sigg_name:<8} | OK     | {urban_result['status']:<6} | "
              f"{'-':>6} | {'-':>6} | {'-':>5} | {elapsed:>7.1f}")
        sigungu_summary.append({
            'sigg_name': sigg_name, 'status_1': 'OK',
            'status_2': urban_result['status'], 'total_in_bbox': 0,
            'matched': 0, 'avoid': 0, 'elapsed': elapsed,
        })
        continue
    
    # 회피 대상 추출
    avoid_features = [
        f for f in urban_result['features']
        if f['properties'].get('uname') in AVOID_UNAMES
    ]
    
    # 군구 정보를 properties에 보강 (이후 통합 GeoJSON에서 출처 추적용)
    for f in avoid_features:
        f['properties']['_target_sigg'] = sigg_name
    
    all_avoid_features.extend(avoid_features)
    all_features_by_sigungu[sigg_name] = urban_result['features']
    
    sigungu_summary.append({
        'sigg_name': sigg_name,
        'status_1': 'OK', 'status_2': 'OK',
        'total_in_bbox': urban_result['total_in_bbox'],
        'matched': urban_result['matched'],
        'avoid': len(avoid_features),
        'elapsed': elapsed,
    })
    
    print(f"{sigg_name:<8} | OK     | OK     | "
          f"{urban_result['total_in_bbox']:>6,} | {urban_result['matched']:>6,} | "
          f"{len(avoid_features):>5} | {elapsed:>7.1f}")

total_elapsed = time.time() - start_time

print("-" * 70)
print(f"\n[전체 소요] {total_elapsed:.1f}초 ({total_elapsed/60:.1f}분)")
print(f"[회피 대상 총합] {len(all_avoid_features)}개 폴리곤\n")

[수집 시작] 인천광역시 9개 군구

시군구      | 1단계    | 2단계    |     전체 |     매칭 |    회피 |   소요(s)
----------------------------------------------------------------------
중구       | OK     | OK     |    591 |    384 |    29 |     0.6
동구       | OK     | OK     |    127 |     58 |    12 |     0.2
미추홀구     | OK     | OK     |    305 |    171 |    15 |     0.5
연수구      | OK     | OK     |    346 |    248 |    11 |     0.4
남동구      | OK     | OK     |    493 |    272 |    14 |     0.6
부평구      | OK     | OK     |    336 |    177 |     7 |     0.4
계양구      | OK     | OK     |    317 |    179 |    12 |     0.5
서구       | OK     | OK     |  1,045 |    505 |    44 |     1.2
강화군      | OK     | OK     |     51 |     50 |     3 |     0.3
----------------------------------------------------------------------

[전체 소요] 4.7초 (0.1분)
[회피 대상 총합] 147개 폴리곤



In [12]:
# 셀 10: 회피 대상 분포 종합 통계
from collections import Counter

print(f"[회피 대상 uname별 분포 (인천 9개 군구 통합)]")
print("=" * 50)
avoid_uname_counter = Counter(f['properties']['uname'] for f in all_avoid_features)
for uname, count in avoid_uname_counter.most_common():
    code = AVOID_UNAMES.get(uname, "?")
    print(f"  [{code}] {uname}: {count}개")

print(f"\n[군구별 회피 대상 분포]")
print("=" * 50)
print(f"{'시군구':<8} | {'UQA310':>6} | {'UQA320':>6} | {'UQA330':>6} | {'합계':>5}")
print("-" * 50)

# 군구별 + UQA 코드별 cross-tabulation
sigg_uname_count = {}
for f in all_avoid_features:
    sigg = f['properties']['_target_sigg']
    uname = f['properties']['uname']
    sigg_uname_count.setdefault(sigg, Counter())[uname] += 1

total_per_code = Counter()
for sigg in INCHEON_DISTRICTS:
    counts = sigg_uname_count.get(sigg, Counter())
    c310 = counts.get("전용공업지역", 0)
    c320 = counts.get("일반공업지역", 0)
    c330 = counts.get("준공업지역", 0)
    total = c310 + c320 + c330
    total_per_code['UQA310'] += c310
    total_per_code['UQA320'] += c320
    total_per_code['UQA330'] += c330
    print(f"{sigg:<8} | {c310:>6} | {c320:>6} | {c330:>6} | {total:>5}")

print("-" * 50)
print(f"{'합계':<8} | {total_per_code['UQA310']:>6} | "
      f"{total_per_code['UQA320']:>6} | {total_per_code['UQA330']:>6} | "
      f"{sum(total_per_code.values()):>5}")

[회피 대상 uname별 분포 (인천 9개 군구 통합)]
  [UQA330] 준공업지역: 96개
  [UQA320] 일반공업지역: 46개
  [UQA310] 전용공업지역: 5개

[군구별 회피 대상 분포]
시군구      | UQA310 | UQA320 | UQA330 |    합계
--------------------------------------------------
중구       |      0 |      4 |     25 |    29
동구       |      0 |      6 |      6 |    12
미추홀구     |      0 |      4 |     11 |    15
연수구      |      0 |      0 |     11 |    11
남동구      |      5 |      4 |      5 |    14
부평구      |      0 |      2 |      5 |     7
계양구      |      0 |      3 |      9 |    12
서구       |      0 |     22 |     22 |    44
강화군      |      0 |      1 |      2 |     3
--------------------------------------------------
합계       |      5 |     46 |     96 |   147


### GeoJSON 저장

In [13]:
# 셀 11: 회피 대상 폴리곤 GeoJSON 저장
import json
from pathlib import Path
from datetime import datetime

# 출력 디렉토리 (작업 폴더 내 ./output)
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

today_str = datetime.now().strftime("%y%m%d")
geojson_path = OUTPUT_DIR / f"incheon_avoid_zones_v1_{today_str}.geojson"

# GeoJSON FeatureCollection 구조 구성
avoid_geojson = {
    "type": "FeatureCollection",
    "name": "incheon_avoid_zones_industrial",
    "crs": {
        "type": "name",
        "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"},  # EPSG:4326의 GeoJSON 표준 표기
    },
    "metadata": {
        "source": "V-World LT_C_UQ111 (도시지역)",
        "target": "인천광역시 9개 군구 (옹진군 제외)",
        "avoid_categories": list(AVOID_UNAMES.keys()),
        "feature_count": len(all_avoid_features),
        "generated_at": datetime.now().isoformat(timespec='seconds'),
    },
    "features": [],
}

# 각 feature를 GeoJSON Feature로 변환 + UQA 코드 명시
for f in all_avoid_features:
    uname = f['properties'].get('uname', '')
    uqa_code = AVOID_UNAMES.get(uname, "UNKNOWN")
    
    avoid_geojson['features'].append({
        "type": "Feature",
        "geometry": f['geometry'],
        "properties": {
            "uname": uname,
            "uqa_code": uqa_code,
            "sido_name": f['properties'].get('sido_name', ''),
            "sigg_name": f['properties'].get('sigg_name', ''),
            "dnum": f['properties'].get('dnum', ''),
            "dyear": f['properties'].get('dyear', ''),
        },
    })

# 파일 저장 (UTF-8, 한글 보존)
with open(geojson_path, 'w', encoding='utf-8') as fp:
    json.dump(avoid_geojson, fp, ensure_ascii=False, indent=2)

print(f"[GeoJSON 저장 완료]")
print(f"  파일: {geojson_path.resolve()}")
print(f"  크기: {geojson_path.stat().st_size / 1024:.1f} KB")
print(f"  features: {len(avoid_geojson['features'])}개")

# 저장 무결성 검증 — 다시 읽어서 카운트 확인
with open(geojson_path, 'r', encoding='utf-8') as fp:
    reloaded = json.load(fp)
print(f"\n[저장 무결성 검증]")
print(f"  재로드 features: {len(reloaded['features'])}개")
print(f"  metadata: {reloaded['metadata']}")

[GeoJSON 저장 완료]
  파일: D:\yyeong\ploggo\algorithm-test\output\incheon_avoid_zones_v1_260605.geojson
  크기: 1719.4 KB
  features: 147개

[저장 무결성 검증]
  재로드 features: 147개
  metadata: {'source': 'V-World LT_C_UQ111 (도시지역)', 'target': '인천광역시 9개 군구 (옹진군 제외)', 'avoid_categories': ['전용공업지역', '일반공업지역', '준공업지역'], 'feature_count': 147, 'generated_at': '2026-06-05T18:30:38'}


In [14]:
# 셀 12: folium 시각화 — 인천 9개 군구 + 회피 대상 색상 구분
import folium

# 인천 대략 중심
INCHEON_CENTER = [37.4563, 126.7052]

m = folium.Map(
    location=INCHEON_CENTER,
    zoom_start=11,
    tiles='OpenStreetMap',
)

# UQA 코드별 색상 (산업 강도가 높을수록 진한 빨강 계열)
COLORS = {
    "UQA310": "#8B0000",  # 전용공업 — 어두운 빨강
    "UQA320": "#E74C3C",  # 일반공업 — 빨강
    "UQA330": "#F39C12",  # 준공업 — 주황
}

# 레이어 1: 시군구 행정구역 경계선 (배경)
sigungu_layer = folium.FeatureGroup(name="시군구 경계", show=True)
for sigg_name, feature in sigungu_polygons.items():
    folium.GeoJson(
        feature['geometry'],
        style_function=lambda x: {
            'color': '#444444',
            'weight': 2,
            'fillOpacity': 0.0,
        },
        tooltip=folium.Tooltip(f"인천광역시 {sigg_name}"),
    ).add_to(sigungu_layer)
sigungu_layer.add_to(m)

# 레이어 2~4: UQA 코드별 회피 폴리곤
for code in ["UQA310", "UQA320", "UQA330"]:
    uname_for_code = next(u for u, c in AVOID_UNAMES.items() if c == code)
    count_for_code = sum(1 for f in avoid_geojson['features'] 
                         if f['properties']['uqa_code'] == code)
    layer_name = f"{code} {uname_for_code} ({count_for_code}개)"
    layer = folium.FeatureGroup(name=layer_name, show=True)
    
    color = COLORS[code]
    for f in avoid_geojson['features']:
        if f['properties']['uqa_code'] != code:
            continue
        popup_html = (
            f"<b>{f['properties']['sigg_name']}</b><br>"
            f"[{code}] {f['properties']['uname']}<br>"
            f"갱신년도: {f['properties']['dyear']}"
        )
        folium.GeoJson(
            f['geometry'],
            style_function=lambda x, c=color: {
                'color': c,
                'weight': 1,
                'fillColor': c,
                'fillOpacity': 0.55,
            },
            popup=folium.Popup(popup_html, max_width=260),
            tooltip=f"{code} · {f['properties']['sigg_name']}",
        ).add_to(layer)
    layer.add_to(m)

# 레이어 컨트롤
folium.LayerControl(collapsed=False).add_to(m)

# 범례
legend_html = f'''
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 1000;
            background: white; padding: 12px 14px; border: 2px solid #555;
            border-radius: 4px; font-size: 12px;
            font-family: 'Malgun Gothic', 'Noto Sans KR', sans-serif;
            box-shadow: 2px 2px 6px rgba(0,0,0,0.2);">
  <b style="font-size: 13px;">회피 영역 범례</b><br>
  <span style="display:inline-block; width:18px; height:12px; vertical-align: middle;
               background:{COLORS['UQA310']}; opacity:0.55;"></span>
  &nbsp;UQA310 전용공업지역<br>
  <span style="display:inline-block; width:18px; height:12px; vertical-align: middle;
               background:{COLORS['UQA320']}; opacity:0.55;"></span>
  &nbsp;UQA320 일반공업지역<br>
  <span style="display:inline-block; width:18px; height:12px; vertical-align: middle;
               background:{COLORS['UQA330']}; opacity:0.55;"></span>
  &nbsp;UQA330 준공업지역<br>
  <span style="display:inline-block; width:18px; height:12px; vertical-align: middle;
               background:none; border:2px solid #444;"></span>
  &nbsp;시군구 경계
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# HTML 파일로 저장 (브라우저에서 단독 열기 가능)
html_path = OUTPUT_DIR / f"incheon_avoid_zones_v1_{today_str}.html"
m.save(str(html_path))
print(f"[시각화 HTML 저장]")
print(f"  파일: {html_path.resolve()}")
print(f"  크기: {html_path.stat().st_size / 1024:.1f} KB")
print(f"\n아래 인라인 지도에서 직접 확인 가능. 외부에서 보려면 HTML 파일을 브라우저로 열 것.")

m  # 셀 마지막 줄에 두면 VS Code Jupyter에서 인라인 렌더링됨

[시각화 HTML 저장]
  파일: D:\yyeong\ploggo\algorithm-test\output\incheon_avoid_zones_v1_260605.html
  크기: 2284.7 KB

아래 인라인 지도에서 직접 확인 가능. 외부에서 보려면 HTML 파일을 브라우저로 열 것.


## 핫스팟이 회피 영역을 지나는가?(카운트)

In [15]:
# 셀 13: 단계 5 진입 — 회피 영역 교차 검증
# 검증 옵션:
#   X. 경유지(핫스팟) 회피 — 셀 16(5-3), 셀 18(5-5)에서 검증
#   Y. polyline 회피      — 셀 19(5-6)에서 검증
# 입력 자산:
#   - 회피 영역 GeoJSON: ./output/incheon_avoid_zones_v1_260601.geojson (147개)
#   - 핫스팟 데이터:    ./hotspots_CCTV_20260510.csv (1,535곳)
#   - 알고리즘 결과:    ./algorithm_test_results_v1.csv (27행, 18개 Greedy+BF 사용)
print("단계 5 진입. 검증 옵션 X + Y.")

단계 5 진입. 검증 옵션 X + Y.


In [16]:
# 셀 14 (5-1): 회피 영역 GeoJSON 로드 및 shapely 변환
# 본 셀은 단독 실행 가능. 단계 4 변수(OUTPUT_DIR, today_str) 의존성 없음.

import json
from pathlib import Path
from shapely.geometry import shape, Point, LineString
from shapely.ops import unary_union
from shapely.prepared import prep

# --- self-contained 블록 (변경점) ---
# OUTPUT_DIR 자체 정의
OUTPUT_DIR = Path("./output")

# GeoJSON 파일 자동 탐색 (날짜 변수 의존성 제거)
geojson_candidates = sorted(OUTPUT_DIR.glob("incheon_avoid_zones_v1_*.geojson"))
if not geojson_candidates:
    raise FileNotFoundError(
        f"회피 영역 GeoJSON을 찾을 수 없다.\n"
        f"검색 경로: {OUTPUT_DIR.resolve()}/incheon_avoid_zones_v1_*.geojson\n"
        f"단계 4 셀 11을 먼저 실행하거나, 파일을 ./output/ 폴더에 배치할 것."
    )
geojson_path_loaded = geojson_candidates[-1]  # 가장 최근 파일 선택
print(f"[로드 대상 GeoJSON] {geojson_path_loaded.name}")
# --- self-contained 블록 끝 ---

with open(geojson_path_loaded, 'r', encoding='utf-8') as fp:
    avoid_geojson_loaded = json.load(fp)

# shapely 폴리곤 리스트 변환 + 메타 정보 보존
avoid_polygons = []
for idx, f in enumerate(avoid_geojson_loaded['features']):
    avoid_polygons.append({
        'idx': idx,
        'geom': shape(f['geometry']),
        'uname': f['properties']['uname'],
        'uqa_code': f['properties']['uqa_code'],
        'sigg_name': f['properties']['sigg_name'],
        'dnum': f['properties'].get('dnum', ''),
    })

# 회피 영역 전체 합집합 — 5-3·5-5의 빠른 contains 검사용
avoid_union = unary_union([p['geom'] for p in avoid_polygons])
avoid_union_prepared = prep(avoid_union)  # 약 10배 빠른 contains 검사

print(f"\n[shapely 변환 완료]")
print(f"  개별 폴리곤: {len(avoid_polygons)}개")
print(f"  통합 영역 타입: {avoid_union.geom_type}")
if avoid_union.geom_type == 'MultiPolygon':
    print(f"  통합 후 분리된 영역 수: {len(avoid_union.geoms)}")
print(f"  통합 영역 BBOX (lon_min, lat_min, lon_max, lat_max):")
print(f"    {tuple(round(v, 5) for v in avoid_union.bounds)}")

# UQA 코드별 합집합도 별도 저장 (5-7 결과 분류용)
avoid_union_by_code = {}
for code in ["UQA310", "UQA320", "UQA330"]:
    code_geoms = [p['geom'] for p in avoid_polygons if p['uqa_code'] == code]
    if code_geoms:
        avoid_union_by_code[code] = unary_union(code_geoms)

print(f"\n[UQA 코드별 합집합 생성]")
for code, union in avoid_union_by_code.items():
    sub_count = len(union.geoms) if union.geom_type == 'MultiPolygon' else 1
    print(f"  {code}: {union.geom_type}, 분리 영역 {sub_count}개")

[로드 대상 GeoJSON] incheon_avoid_zones_v1_260605.geojson

[shapely 변환 완료]
  개별 폴리곤: 147개
  통합 영역 타입: MultiPolygon
  통합 후 분리된 영역 수: 75
  통합 영역 BBOX (lon_min, lat_min, lon_max, lat_max):
    (126.47064, 37.34181, 126.76624, 37.76786)

[UQA 코드별 합집합 생성]
  UQA310: MultiPolygon, 분리 영역 5개
  UQA320: MultiPolygon, 분리 영역 30개
  UQA330: MultiPolygon, 분리 영역 91개


In [17]:
# 셀 15 (5-2): 핫스팟 데이터 로드 및 컬럼 구조 확인
# 핫스팟 CSV는 작업 폴더(BASE_DIR) 직속에 있어야 한다

BASE_DIR = Path.cwd()
hotspots_csv_path = BASE_DIR / "hotspots_CCTV_20260510.csv"

if not hotspots_csv_path.exists():
    raise FileNotFoundError(
        f"핫스팟 CSV 파일을 찾을 수 없다.\n"
        f"확인한 경로:\n  {hotspots_csv_path}\n"
        f"파일을 작업 폴더에 복사할 것."
    )

df_hotspots = pd.read_csv(hotspots_csv_path, encoding='utf-8-sig')

print(f"[핫스팟 데이터 로드]")
print(f"  파일: {hotspots_csv_path.resolve()}")
print(f"  크기: {hotspots_csv_path.stat().st_size / 1024:.1f} KB")
print(f"  행 수: {len(df_hotspots):,}")
print(f"  컬럼 수: {len(df_hotspots.columns)}")

print(f"\n[컬럼 목록 — 5-3 작성에 필요]")
for i, col in enumerate(df_hotspots.columns):
    dtype = str(df_hotspots[col].dtype)
    sample = df_hotspots[col].iloc[0]
    if isinstance(sample, str) and len(sample) > 40:
        sample = sample[:40] + "..."
    print(f"  {i:2d}. {col:<20s} ({dtype:<10s}) 예: {sample}")

print(f"\n[첫 3행 미리보기]")
print(df_hotspots.head(3).to_string())

print(f"\n[중요 컬럼 후보 확인 — 위경도/ID/군구]")
for keyword in ['lat', 'lon', 'id', 'district', 'sigg', 'cctv', 'neighbor', 'S']:
    matches = [c for c in df_hotspots.columns if keyword.lower() in c.lower()]
    print(f"  '{keyword}' 매칭: {matches}")

[핫스팟 데이터 로드]
  파일: D:\yyeong\ploggo\algorithm-test\hotspots_CCTV_20260510.csv
  크기: 331.9 KB
  행 수: 1,535
  컬럼 수: 15

[컬럼 목록 — 5-3 작성에 필요]
   0. hotspot_id           (str       ) 예: bupyeong_wydj5j5q_001
   1. source_type          (str       ) 예: district_file
   2. source_district      (str       ) 예: bupyeong
   3. latitude             (float64   ) 예: 37.4699291
   4. longitude            (float64   ) 예: 126.6988679
   5. geohash7             (str       ) 예: wydj5j5
   6. geohash8             (str       ) 예: wydj5j5q
   7. dong                 (str       ) 예: 십정2동
   8. address_road         (str       ) 예: 백범로 478번길 19
   9. address_jibun        (str       ) 예: nan
  10. address_raw          (str       ) 예: 백범로478번길 19 (일산슈퍼 앞, 석정공원 코너)
  11. cctv_count           (int64     ) 예: 1
  12. install_years        (str       ) 예: [2009]
  13. cctv_type            (str       ) 예: fixed
  14. relocation_history   (str       ) 예: [{'date': '2016.10.26.', 'note': '이전'}]

[첫 3행 미리보기]
           

In [18]:
# 셀 16 (5-3): 전체 핫스팟 1,535곳 중 회피 영역 내부 카운트
# - 옵션 X 1차 검증: 핫스팟 자체가 회피 영역에 위치하는 비율
# - 경계선 위의 점도 회피 영역으로 분류 (intersects 사용, 보수적 접근)
from shapely.geometry import Point

def check_avoid_zone(row):
    """
    Returns (in_zone: bool, uqa_code: str | None)
    """
    pt = Point(row['longitude'], row['latitude'])  # shapely는 (x=lon, y=lat) 순서
    if not avoid_union_prepared.intersects(pt):
        return False, None
    # UQA 코드 식별 (UQA310 → UQA320 → UQA330 순서로 검사)
    for code, union in avoid_union_by_code.items():
        if union.intersects(pt):
            return True, code
    return True, 'UNKNOWN'

# 1,535곳 전체 검사 (예상 소요: 1초 미만, prepared geometry 덕분)
result = df_hotspots.apply(check_avoid_zone, axis=1)
df_hotspots['in_avoid_zone'] = result.apply(lambda x: x[0])
df_hotspots['avoid_uqa_code'] = result.apply(lambda x: x[1])

# 결과 집계
total = len(df_hotspots)
inside = df_hotspots['in_avoid_zone'].sum()

print(f"[전체 핫스팟 회피 영역 검사 결과]")
print(f"  전체 핫스팟: {total:,}곳")
print(f"  회피 영역 내부: {inside}곳 ({inside/total*100:.2f}%)")

if inside == 0:
    print(f"\n→ 핫스팟 1,535곳 중 회피 영역 내부 0건.")
    print(f"  옵션 X(경유지 회피) 1차 검증 결론: 회피 핫스팟이 데이터셋에 존재하지 않음.")
    print(f"  단, 폴리곤 경계 인접 핫스팟 가능성을 위해 5-5·5-6 추가 검증 필요.")
else:
    # 군구별 분포
    print(f"\n[회피 영역 내부 핫스팟 — 군구별]")
    grouped = df_hotspots[df_hotspots['in_avoid_zone']].groupby('source_district').size()
    for dist, count in grouped.sort_values(ascending=False).items():
        total_in_dist = (df_hotspots['source_district'] == dist).sum()
        pct = count / total_in_dist * 100
        print(f"  {dist:<10s}: {count:>3}곳 / 전체 {total_in_dist:>3}곳 ({pct:>5.1f}%)")
    
    # UQA 코드별 분포
    print(f"\n[회피 영역 내부 핫스팟 — UQA 코드별]")
    code_grouped = df_hotspots[df_hotspots['in_avoid_zone']].groupby('avoid_uqa_code').size()
    for code, count in code_grouped.sort_values(ascending=False).items():
        print(f"  {code}: {count}곳")
    
    # 군구 × UQA 코드 cross-tabulation
    print(f"\n[군구 × UQA 코드 cross-tabulation]")
    cross = pd.crosstab(
        df_hotspots[df_hotspots['in_avoid_zone']]['source_district'],
        df_hotspots[df_hotspots['in_avoid_zone']]['avoid_uqa_code'],
        margins=True, margins_name='합계'
    )
    print(cross)
    
    # 회피 영역 내부 핫스팟의 CCTV 카메라 합계 (참고)
    avoid_cctv_total = df_hotspots[df_hotspots['in_avoid_zone']]['cctv_count'].sum()
    print(f"\n[회피 영역 내부 CCTV 카메라 합계] {avoid_cctv_total}대")
    
    # 회피 영역 내부 핫스팟의 dong 분포 (지역명 확인)
    print(f"\n[회피 영역 내부 핫스팟 — 행정동 상위 10개]")
    dong_grouped = df_hotspots[df_hotspots['in_avoid_zone']]['dong'].value_counts().head(10)
    for dong, count in dong_grouped.items():
        print(f"  {dong}: {count}곳")

[전체 핫스팟 회피 영역 검사 결과]
  전체 핫스팟: 1,535곳
  회피 영역 내부: 58곳 (3.78%)

[회피 영역 내부 핫스팟 — 군구별]
  seogu     :  21곳 / 전체 379곳 (  5.5%)
  gyeyang   :  15곳 / 전체 292곳 (  5.1%)
  donggu    :   8곳 / 전체  48곳 ( 16.7%)
  namdong   :   7곳 / 전체 155곳 (  4.5%)
  junggu    :   5곳 / 전체  74곳 (  6.8%)
  bupyeong  :   2곳 / 전체  42곳 (  4.8%)

[회피 영역 내부 핫스팟 — UQA 코드별]
  UQA330: 45곳
  UQA320: 13곳

[군구 × UQA 코드 cross-tabulation]
avoid_uqa_code   UQA320  UQA330  합계
source_district                    
bupyeong              0       2   2
donggu                5       3   8
gyeyang               0      15  15
junggu                1       4   5
namdong               3       4   7
seogu                 4      17  21
합계                   13      45  58

[회피 영역 내부 CCTV 카메라 합계] 64대

[회피 영역 내부 핫스팟 — 행정동 상위 10개]
  가좌1동: 12곳
  효성동: 8곳
  작전동: 7곳
  만석동: 3곳
  항동7가: 3곳
  논현고잔동: 3곳
  가좌3동: 3곳
  청천1동: 2곳
  화수2동: 2곳
  송현3동: 2곳


In [ ]:
# 삭제해도됨: 시설명 매칭 한계로 본 노트북에서 미사용. 단계 6 OD도 좌표 직접 지정 예정

# # 셀 17 (5-4 (1)): Naver API 자격증명 로드 + Geocoding 클라이언트
# # .env에 NAVER_MAPS_CLIENT_ID, NAVER_MAPS_CLIENT_SECRET 추가되어 있어야 함

# import os
# import time
# import requests
# import json
# from dotenv import load_dotenv

# # .env 재로드 (단계 4 실행 후 .env 수정한 경우 필수)
# load_dotenv(override=True)

# NAVER_CLIENT_ID = os.getenv('NAVER_MAPS_CLIENT_ID')
# NAVER_CLIENT_SECRET = os.getenv('NAVER_MAPS_CLIENT_SECRET')

# if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
#     raise RuntimeError(
#         "Naver API 자격증명을 찾을 수 없다.\n"
#         ".env 파일에 NAVER_MAPS_CLIENT_ID, NAVER_MAPS_CLIENT_SECRET가 있는지 확인할 것."
#     )

# print(f"[Naver API 자격증명 로드 완료]")
# print(f"  Client ID: {NAVER_CLIENT_ID[:6]}...{NAVER_CLIENT_ID[-4:]}")
# print(f"  Client Secret: 마스킹됨 ({len(NAVER_CLIENT_SECRET)}자)")

# # Naver Geocoding API
# # NCP 2024 이후 표준 endpoint: maps.apigw.ntruss.com
# # 만약 401 발생 시 routeTest_v3의 endpoint와 헤더 형식 확인 필요
# NAVER_GEOCODE_URL = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"

# def geocode_address(address, debug=False):
#     """
#     Naver Geocoding API로 주소를 좌표로 변환.
#     Returns: {'lat': float, 'lon': float, 'matched_address': str, 'status': str}
#     """
#     headers = {
#         "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
#         "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET,
#         "Accept": "application/json",
#     }
#     params = {"query": address}
    
#     try:
#         r = requests.get(NAVER_GEOCODE_URL, headers=headers, params=params, timeout=10)
#     except requests.exceptions.RequestException as e:
#         return {'lat': None, 'lon': None, 'matched_address': None, 
#                 'status': f'ERROR: {type(e).__name__}: {e}'}
    
#     if debug:
#         print(f"  [HTTP {r.status_code}] Content-Type: {r.headers.get('Content-Type', 'N/A')}")
#         print(f"  [응답 본문 앞부분] {r.text[:400]}")
    
#     if r.status_code != 200:
#         return {'lat': None, 'lon': None, 'matched_address': None,
#                 'status': f'ERROR: HTTP {r.status_code}, body: {r.text[:200]}'}
    
#     try:
#         result = r.json()
#     except json.JSONDecodeError:
#         return {'lat': None, 'lon': None, 'matched_address': None,
#                 'status': f'ERROR: JSONDecodeError, body: {r.text[:200]}'}
    
#     if result.get('status') != 'OK':
#         return {'lat': None, 'lon': None, 'matched_address': None,
#                 'status': f"API status: {result.get('status')}, msg: {result.get('errorMessage', '')}"}
    
#     addresses = result.get('addresses', [])
#     if not addresses:
#         return {'lat': None, 'lon': None, 'matched_address': None, 'status': 'NO_RESULT'}
    
#     first = addresses[0]
#     try:
#         lon = float(first['x'])  # Naver는 x=경도, y=위도
#         lat = float(first['y'])
#     except (KeyError, ValueError, TypeError) as e:
#         return {'lat': None, 'lon': None, 'matched_address': None,
#                 'status': f'ERROR: 좌표 파싱 실패 - {e}'}
    
#     matched = first.get('roadAddress') or first.get('jibunAddress') or ''
#     return {'lat': lat, 'lon': lon, 'matched_address': matched, 'status': 'OK'}


# # 동작 검증 — 미추홀구청으로 1회 호출
# print(f"\n[Geocoding 동작 검증] '인천 미추홀구청'")
# test_result = geocode_address("인천 미추홀구청", debug=True)
# for k, v in test_result.items():
#     print(f"  {k}: {v}")

[Naver API 자격증명 로드 완료]
  Client ID: zrs7ik...kup9
  Client Secret: 마스킹됨 (40자)

[Geocoding 동작 검증] '인천 미추홀구청'
  [HTTP 200] Content-Type: application/json
  [응답 본문 앞부분] {"status":"OK","meta":{"totalCount":0,"count":0},"addresses":[],"errorMessage":""}
  lat: None
  lon: None
  matched_address: None
  status: NO_RESULT


In [20]:
# 셀 18 (5-4 (2)) [재작성]: OD 9쌍 좌표 — routeTest_v3 동기화 상수
# routeTest_v3의 Google Geocoding 결과를 그대로 사용하여 정합성 확보.
# 본 노트북은 검증용이므로 좌표 상수 입력을 허용한다 (routeTest_v3와 다른 정책).

import math

# routeTest_v3 / OD_CANDIDATES_V2 직접 복사 (좌표는 (lat, lon) 순서)
OD_CANDIDATES_V2 = {
    'ganghwa':  {'O': (37.706505, 126.443398), 'D': (37.732838, 126.517697)},
    'junggu':   {'O': (37.473565, 126.621778), 'D': (37.474748, 126.603258)},
    'donggu':   {'O': (37.470733, 126.641914), 'D': (37.487780, 126.621754)},
    'michuhol': {'O': (37.455015, 126.667435), 'D': (37.435062, 126.690721)},
    'seogu':    {'O': (37.545115, 126.676062), 'D': (37.552748, 126.650239)},
    'yeonsu':   {'O': (37.391182, 126.645092), 'D': (37.418275, 126.661122)},
    'namdong':  {'O': (37.447245, 126.731453), 'D': (37.447769, 126.700089)},
    'bupyeong': {'O': (37.507046, 126.721858), 'D': (37.482179, 126.703038)},
    'gyeyang':  {'O': (37.537367, 126.737744), 'D': (37.530279, 126.722516)},
}

# 셀 19에서 사용할 통일 자료구조명으로 변환
od_coords = {}
for district, pts in OD_CANDIDATES_V2.items():
    od_coords[district] = {
        'O': pts['O'],
        'D': pts['D'],
        'O_addr_query': None,    # Geocoding 미호출
        'D_addr_query': None,
        'O_addr_matched': None,
        'D_addr_matched': None,
    }

# Haversine 직선거리 계산 함수
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2)
    return 2 * R * math.asin(math.sqrt(a))

# 출력 + 핸드오프 문서 5-1표 직선거리와 자동 대조
expected_distances = {
    'ganghwa': 7.16, 'junggu': 1.64, 'donggu': 2.60, 'michuhol': 1.72,
    'seogu': 2.43, 'yeonsu': 3.33, 'namdong': 2.77, 'bupyeong': 3.23, 'gyeyang': 1.56,
}

print(f"[OD 9쌍 좌표 — routeTest_v3 동기화 완료]\n")
print(f"{'군구':<10} | {'O 좌표':<22} | {'D 좌표':<22} | {'실측':>6} | {'문서':>5} | {'차이':>6}")
print("-" * 85)

max_diff = 0.0
for district, coords in od_coords.items():
    o = coords['O']
    d = coords['D']
    actual = haversine_km(o[0], o[1], d[0], d[1])
    expected = expected_distances[district]
    diff = abs(actual - expected)
    max_diff = max(max_diff, diff)
    flag = "" if diff < 0.05 else " ← 차이 큼"
    print(f"{district:<10} | ({o[0]:.4f}, {o[1]:.4f}) | ({d[0]:.4f}, {d[1]:.4f}) | "
          f"{actual:>5.2f} | {expected:>5.2f} | {diff:>+5.2f}{flag}")

print(f"\n[정합성 검증] 최대 차이: {max_diff:.3f} km")
if max_diff < 0.05:
    print(f"  → 모든 OD가 핸드오프 문서 5-1표 직선거리와 일치 (오차 50m 이하).")
    print(f"  → routeTest_v3 동기화 완료. 셀 19 진입 가능.")
else:
    print(f"  → 일부 OD에서 직선거리 차이 발견. 좌표 입력 재확인 권고.")

[OD 9쌍 좌표 — routeTest_v3 동기화 완료]

군구         | O 좌표                   | D 좌표                   |     실측 |    문서 |     차이
-------------------------------------------------------------------------------------
ganghwa    | (37.7065, 126.4434) | (37.7328, 126.5177) |  7.16 |  7.16 | +0.00
junggu     | (37.4736, 126.6218) | (37.4747, 126.6033) |  1.64 |  1.64 | +0.00
donggu     | (37.4707, 126.6419) | (37.4878, 126.6218) |  2.60 |  2.60 | +0.00
michuhol   | (37.4550, 126.6674) | (37.4351, 126.6907) |  3.02 |  1.72 | +1.30 ← 차이 큼
seogu      | (37.5451, 126.6761) | (37.5527, 126.6502) |  2.43 |  2.43 | +0.00
yeonsu     | (37.3912, 126.6451) | (37.4183, 126.6611) |  3.33 |  3.33 | +0.00
namdong    | (37.4472, 126.7315) | (37.4478, 126.7001) |  2.77 |  2.77 | +0.00
bupyeong   | (37.5070, 126.7219) | (37.4822, 126.7030) |  3.23 |  3.23 | +0.00
gyeyang    | (37.5374, 126.7377) | (37.5303, 126.7225) |  1.56 |  1.56 | +0.00

[정합성 검증] 최대 차이: 1.305 km
  → 일부 OD에서 직선거리 차이 발견. 좌표 입력 재확인 권고.


In [21]:
# 셀 19 (5-4 (3)) [상단 부분만 수정]: algorithm_test_results_v1.csv 파싱

# 검색 경로 확장 — 여러 위치 시도
BASE_DIR = Path.cwd()
algo_csv_candidates = [
    BASE_DIR / "algorithm_test_results_v1.csv",
    BASE_DIR.parent / "algorithm_test_results_v1.csv",
]
algo_csv_path = None
for candidate in algo_csv_candidates:
    if candidate.exists():
        algo_csv_path = candidate
        break

if algo_csv_path is None:
    raise FileNotFoundError(
        f"algorithm_test_results_v1.csv를 찾을 수 없다.\n"
        f"검색한 경로:\n" + "\n".join(f"  - {c}" for c in algo_csv_candidates) + "\n"
        f"파일을 작업 폴더에 복사할 것."
    )

df_algo = pd.read_csv(algo_csv_path, encoding='utf-8-sig')
print(f"[알고리즘 결과 CSV 로드]")
print(f"  파일: {algo_csv_path.resolve()}")
print(f"  전체 행: {len(df_algo)}")

# (이하 코드는 기존 셀 19와 동일 — Greedy + BruteForce 필터링, hotspot_coords lookup, scenarios 구성)
# Greedy + BruteForce만 (Clustering 제외)
df_algo_filtered = df_algo[df_algo['algorithm'].isin(['Greedy', 'BruteForce'])].copy()
print(f"  필터링 (Greedy + BruteForce): {len(df_algo_filtered)}행")

# hotspot_id → (lat, lon) lookup 사전
hotspot_coords = {
    row['hotspot_id']: (row['latitude'], row['longitude'])
    for _, row in df_hotspots.iterrows()
}
print(f"  핫스팟 lookup 사전: {len(hotspot_coords):,}개")

# 18개 시나리오 구성
print(f"\n[18개 시나리오 visit_order 파싱]")
scenarios = []
parse_errors = []

for _, row in df_algo_filtered.iterrows():
    district = row['district']
    algorithm = row['algorithm']
    visit_order_str = row['visit_order']
    
    # 파이프로 구분된 핫스팟 ID 리스트
    waypoint_ids = visit_order_str.split('|')
    
    # 좌표 lookup
    waypoint_coords = []
    missing = []
    for hid in waypoint_ids:
        if hid in hotspot_coords:
            waypoint_coords.append(hotspot_coords[hid])
        else:
            missing.append(hid)
    
    if missing:
        parse_errors.append({
            'district': district, 'algorithm': algorithm,
            'missing_ids': missing,
        })
        continue
    
    # OD 좌표 확보 확인
    od = od_coords.get(district)
    if od is None or od.get('O') is None:
        parse_errors.append({
            'district': district, 'algorithm': algorithm,
            'reason': 'OD coords missing',
        })
        continue
    
    # 전체 경로: O → 경유지 K개 → D
    route_coords = [od['O']] + waypoint_coords + [od['D']]
    
    scenarios.append({
        'district': district,
        'algorithm': algorithm,
        'k': len(waypoint_ids),  # 일반적으로 3, 후보 부족 시 2
        'waypoint_ids': waypoint_ids,
        'waypoint_coords': waypoint_coords,
        'route_coords': route_coords,
        'O': od['O'],
        'D': od['D'],
        # 알고리즘 결과 원본 지표도 보존
        'score': row['score'],
        'accuracy_pct': row['accuracy_pct'],
        'detour_km': row['detour_km'],
        'mpd_km': row['mpd_km'],
        'total_S': row['total_S'],
    })

print(f"  성공: {len(scenarios)}개 시나리오")
print(f"  실패: {len(parse_errors)}개")

if parse_errors:
    print(f"\n[파싱 실패 케이스]")
    for err in parse_errors:
        print(f"  {err}")

# 시나리오 분포 확인
print(f"\n[시나리오 분포]")
algo_counts = {}
for s in scenarios:
    algo_counts[s['algorithm']] = algo_counts.get(s['algorithm'], 0) + 1
for algo, count in algo_counts.items():
    print(f"  {algo}: {count}개")

# K 값 분포 (후보 부족 시 K=2 fallback 확인)
print(f"\n[경유 핫스팟 수(K) 분포]")
k_counts = {}
for s in scenarios:
    k_counts[s['k']] = k_counts.get(s['k'], 0) + 1
for k, count in sorted(k_counts.items()):
    print(f"  K={k}: {count}개")

# 첫 시나리오 상세 (디버깅용)
if scenarios:
    print(f"\n[시나리오 #1 상세]")
    s = scenarios[0]
    print(f"  district: {s['district']} / algorithm: {s['algorithm']}")
    print(f"  K: {s['k']}, score: {s['score']:.3f}, accuracy: {s['accuracy_pct']:.2f}%")
    print(f"  route ({len(s['route_coords'])}개 점):")
    for i, pt in enumerate(s['route_coords']):
        if i == 0:
            label = 'O'
        elif i == len(s['route_coords']) - 1:
            label = 'D'
        else:
            label = f"W{i} ({s['waypoint_ids'][i-1]})"
        print(f"    {label}: ({pt[0]:.4f}, {pt[1]:.4f})")

[알고리즘 결과 CSV 로드]
  파일: D:\yyeong\ploggo\algorithm-test\algorithm_test_results_v1.csv
  전체 행: 27
  필터링 (Greedy + BruteForce): 18행
  핫스팟 lookup 사전: 1,535개

[18개 시나리오 visit_order 파싱]
  성공: 18개 시나리오
  실패: 0개

[시나리오 분포]
  Greedy: 9개
  BruteForce: 9개

[경유 핫스팟 수(K) 분포]
  K=3: 18개

[시나리오 #1 상세]
  district: ganghwa / algorithm: Greedy
  K: 3, score: 1.913, accuracy: 100.00%
  route (5개 점):
    O: (37.7065, 126.4434)
    W1 (ganghwa_wy9yt9s6_001): (37.7137, 126.4588)
    W2 (ganghwa_wy9yw5uk_001): (37.7264, 126.4807)
    W3 (ganghwa_wy9yw5vs_001): (37.7265, 126.4824)
    D: (37.7328, 126.5177)


### 회피 영역 내부카운트

In [22]:
# 셀 20 (5-5): 18개 시나리오의 경유 핫스팟 회피 영역 내부 카운트
# - 옵션 X 2차 검증 (핵심): 실제 선정된 경유지가 회피 영역에 위치하는가
from shapely.geometry import Point

# 18개 시나리오의 모든 waypoint를 평탄화 (flat DataFrame)
all_waypoints = []
for s in scenarios:
    for i, (wp_id, coord) in enumerate(zip(s['waypoint_ids'], s['waypoint_coords'])):
        all_waypoints.append({
            'district': s['district'],
            'algorithm': s['algorithm'],
            'waypoint_idx': i + 1,        # W1, W2, W3
            'hotspot_id': wp_id,
            'lat': coord[0],
            'lon': coord[1],
        })

df_waypoints = pd.DataFrame(all_waypoints)
print(f"[전체 경유 핫스팟] {len(df_waypoints)}개 ({len(scenarios)}개 시나리오 × K=3)")

# 회피 영역 내부 검사
def check_waypoint(row):
    pt = Point(row['lon'], row['lat'])
    if not avoid_union_prepared.intersects(pt):
        return False, None
    for code, union in avoid_union_by_code.items():
        if union.intersects(pt):
            return True, code
    return True, 'UNKNOWN'

result = df_waypoints.apply(check_waypoint, axis=1)
df_waypoints['in_avoid_zone'] = result.apply(lambda x: x[0])
df_waypoints['avoid_uqa_code'] = result.apply(lambda x: x[1])

# === 1. 전체 결과 ===
total_waypoints = len(df_waypoints)
inside = df_waypoints['in_avoid_zone'].sum()

print(f"\n[옵션 X 2차 검증 결과]")
print(f"  전체 경유 핫스팟 (위치 점 단위): {total_waypoints}개")
print(f"  회피 영역 내부:                  {inside}개 ({inside/total_waypoints*100:.1f}%)")

# === 2. 고유 핫스팟 분석 (Greedy/BF 중복 제거) ===
unique_hotspots = df_waypoints['hotspot_id'].nunique()
unique_in_avoid = df_waypoints[df_waypoints['in_avoid_zone']]['hotspot_id'].nunique()
print(f"\n  고유 핫스팟: {unique_hotspots}개 (54개 중 Greedy/BF 중복 제거)")
print(f"  고유 회피 핫스팟: {unique_in_avoid}개")

# === 3. 결과 분기 ===
if inside == 0:
    print(f"\n{'=' * 60}")
    print(f"[결론] 9 OD × 2 algo = 18 시나리오에서 회피 영역 핫스팟 0건.")
    print(f"{'=' * 60}")
    print(f"Greedy/BF가 S_i 기준으로 회피 영역 핫스팟을 자연 배제함을 확인.")
    print(f"옵션 X 추가 가치는 본 9 OD 표본에서 미관측.")
    print(f"단계 6에서 일반화 검증 필요.")
else:
    # 알고리즘별 비교
    print(f"\n[알고리즘별 회피 영역 핫스팟 선정 비율]")
    for algo in ['Greedy', 'BruteForce']:
        algo_df = df_waypoints[df_waypoints['algorithm'] == algo]
        algo_inside = algo_df['in_avoid_zone'].sum()
        print(f"  {algo:<11s}: {algo_inside}개 / {len(algo_df)}개 ({algo_inside/len(algo_df)*100:.1f}%)")
    
    # 군구별 분포
    print(f"\n[군구별 회피 영역 핫스팟 선정]")
    print(f"{'군구':<10} | {'Greedy':>6} | {'BruteForce':>10} | {'합계':>4}")
    print("-" * 40)
    for district in df_waypoints['district'].unique():
        dist_df = df_waypoints[df_waypoints['district'] == district]
        g_count = dist_df[(dist_df['algorithm'] == 'Greedy') & dist_df['in_avoid_zone']].shape[0]
        b_count = dist_df[(dist_df['algorithm'] == 'BruteForce') & dist_df['in_avoid_zone']].shape[0]
        if g_count + b_count > 0:
            print(f"{district:<10} | {g_count:>6} | {b_count:>10} | {g_count+b_count:>4}")
    
    # UQA 코드별 분포
    print(f"\n[UQA 코드별 회피 영역 핫스팟 분포]")
    code_counter = df_waypoints[df_waypoints['in_avoid_zone']]['avoid_uqa_code'].value_counts()
    for code, count in code_counter.items():
        print(f"  {code}: {count}개")
    
    # 상세 — 회피 영역 내부 핫스팟 전체 출력
    print(f"\n[회피 영역 내부 경유 핫스팟 상세]")
    print(f"{'군구':<10} | {'알고리즘':<10} | {'W':<3} | {'hotspot_id':<28} | {'UQA':<7} | {'좌표'}")
    print("-" * 95)
    inside_df = df_waypoints[df_waypoints['in_avoid_zone']].sort_values(['district', 'algorithm', 'waypoint_idx'])
    for _, row in inside_df.iterrows():
        print(f"{row['district']:<10} | {row['algorithm']:<10} | W{row['waypoint_idx']:<2} | "
              f"{row['hotspot_id']:<28} | {row['avoid_uqa_code']:<7} | "
              f"({row['lat']:.4f}, {row['lon']:.4f})")
    
    # Greedy vs BF 차이 분석
    print(f"\n[Greedy vs BruteForce 차이 분석]")
    g_set = set(df_waypoints[(df_waypoints['algorithm'] == 'Greedy') & df_waypoints['in_avoid_zone']]['hotspot_id'])
    b_set = set(df_waypoints[(df_waypoints['algorithm'] == 'BruteForce') & df_waypoints['in_avoid_zone']]['hotspot_id'])
    common = g_set & b_set
    g_only = g_set - b_set
    b_only = b_set - g_set
    print(f"  공통 회피 핫스팟: {len(common)}개")
    print(f"  Greedy 단독 회피 핫스팟: {len(g_only)}개 — {list(g_only) if g_only else '없음'}")
    print(f"  BF 단독 회피 핫스팟:     {len(b_only)}개 — {list(b_only) if b_only else '없음'}")

[전체 경유 핫스팟] 54개 (18개 시나리오 × K=3)

[옵션 X 2차 검증 결과]
  전체 경유 핫스팟 (위치 점 단위): 54개
  회피 영역 내부:                  0개 (0.0%)

  고유 핫스팟: 33개 (54개 중 Greedy/BF 중복 제거)
  고유 회피 핫스팟: 0개

[결론] 9 OD × 2 algo = 18 시나리오에서 회피 영역 핫스팟 0건.
Greedy/BF가 S_i 기준으로 회피 영역 핫스팟을 자연 배제함을 확인.
옵션 X 추가 가치는 본 9 OD 표본에서 미관측.
단계 6에서 일반화 검증 필요.


## 옵션 Y : polyline이 회피 영역을 통과하는지?

In [23]:
# 셀 21 (재작성): Tmap 보행자 경로안내 API 호출 함수
# Naver Directions 5(차량) → Tmap 보행자 API(도보) 전환

import os
import requests
import json
import time
from dotenv import load_dotenv

# .env 재로드 (TMAP_APP_KEY 새로 추가했으므로 필수)
load_dotenv(override=True)

TMAP_APP_KEY = os.getenv('TMAP_APP_KEY')
if not TMAP_APP_KEY:
    raise RuntimeError(".env에 TMAP_APP_KEY가 없다.")

print(f"[Tmap AppKey 로드 완료]")
print(f"  AppKey: {TMAP_APP_KEY[:6]}...{TMAP_APP_KEY[-4:]}")

TMAP_PEDESTRIAN_URL = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"

def get_pedestrian_route(origin, destination, waypoints=None, debug=False):
    """
    Tmap 보행자 경로안내 API로 도보 polyline 획득.
    Args:
        origin: (lat, lon)
        destination: (lat, lon)
        waypoints: [(lat, lon), ...] 또는 None (최대 5개)
    Returns:
        dict: path([(lat, lon), ...]), distance_m, duration_ms, status
    """
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "appKey": TMAP_APP_KEY,
    }
    payload = {
        "startX": origin[1], "startY": origin[0],
        "endX": destination[1], "endY": destination[0],
        "startName": "출발지", "endName": "도착지",
        "reqCoordType": "WGS84GEO", "resCoordType": "WGS84GEO",
        "searchOption": "0",
    }
    if waypoints:
        if len(waypoints) > 5:
            return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                    'status': f'ERROR: 경유지 {len(waypoints)}개. Tmap 한도 5개 초과'}
        payload["passList"] = "_".join(f"{wp[1]},{wp[0]}" for wp in waypoints)
    
    try:
        r = requests.post(TMAP_PEDESTRIAN_URL, headers=headers, json=payload, timeout=15)
    except requests.exceptions.RequestException as e:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f'ERROR: {type(e).__name__}'}
    
    if debug:
        print(f"  [HTTP {r.status_code}] {r.headers.get('Content-Type', '')}")
        print(f"  [Body 앞부분] {r.text[:400]}")
    
    if r.status_code != 200:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f'ERROR: HTTP {r.status_code}, body: {r.text[:200]}'}
    
    try:
        result = r.json()
    except json.JSONDecodeError:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'ERROR: JSONDecodeError'}
    
    if result.get('type') != 'FeatureCollection':
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f"Invalid response: {str(result)[:200]}"}
    
    features = result.get('features', [])
    if not features:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'NO_FEATURES'}
    
    first_props = features[0].get('properties', {})
    total_distance_m = first_props.get('totalDistance', 0)
    total_time_s = first_props.get('totalTime', 0)
    
    path_lonlat = []
    for f in features:
        geom = f.get('geometry', {})
        if geom.get('type') == 'LineString':
            path_lonlat.extend(geom.get('coordinates', []))
    
    path_latlon = [(pt[1], pt[0]) for pt in path_lonlat]
    
    return {
        'path': path_latlon,
        'distance_m': total_distance_m,
        'duration_ms': total_time_s * 1000,
        'status': 'OK',
    }

# 동작 검증
print(f"\n[Tmap 보행자 경로안내 동작 검증]")
test_s = scenarios[0]
print(f"  시나리오: {test_s['district']} / {test_s['algorithm']}")
test_result = get_pedestrian_route(
    origin=test_s['O'],
    destination=test_s['D'],
    waypoints=test_s['waypoint_coords'],
    debug=True,
)
print(f"\n[결과]")
print(f"  status: {test_result['status']}")
if test_result['status'] == 'OK':
    print(f"  path 좌표 수: {len(test_result['path']):,}개")
    print(f"  도보 거리: {test_result['distance_m']:,}m ({test_result['distance_m']/1000:.2f}km)")
    print(f"  도보 소요: {test_result['duration_ms']/60000:.1f}분")

[Tmap AppKey 로드 완료]
  AppKey: r8mALt...H5Zm

[Tmap 보행자 경로안내 동작 검증]
  시나리오: ganghwa / Greedy
  [HTTP 200] application/json;charset=UTF-8
  [Body 앞부분] {  "type": "FeatureCollection",  "features": [    {"type": "Feature",     "geometry": {       "type": "Point",       "coordinates": [126.44396871542024,37.706656643446884 ]     },     "properties":{      "totalDistance": 9456,      "totalTime": 6879,      "index": 0,      "pointIndex": 0,      "name": "",      "description": "79m 이동",      "direction": "",      "nearPoiName": "",      "nearPoiX": 

[결과]
  status: OK
  path 좌표 수: 373개
  도보 거리: 9,456m (9.46km)
  도보 소요: 114.7분


In [24]:
# 셀 22 (5-6 (2)): 18개 시나리오 polyline 일괄 생성
# 각 호출 사이 0.3초 대기로 NCP rate limit 대비

print(f"[Naver Directions 5 호출 시작 — 18 시나리오]\n")
print(f"{'idx':<4} | {'군구':<10} | {'algo':<10} | {'path점':>7} | {'거리(km)':>9} | {'분':>5} | status")
print("-" * 75)

t_start = time.time()
for i, s in enumerate(scenarios):
    r = get_pedestrian_route(
        origin=s['O'],
        destination=s['D'],
        waypoints=s['waypoint_coords'],
    )
    s['polyline'] = r.get('path')
    s['polyline_distance_m'] = r.get('distance_m', 0)
    s['polyline_duration_ms'] = r.get('duration_ms', 0)
    s['polyline_status'] = r.get('status')
    
    pcount = len(r['path']) if r['path'] else 0
    dist_km = r.get('distance_m', 0) / 1000
    dur_min = r.get('duration_ms', 0) / 60000
    
    print(f"{i+1:<4} | {s['district']:<10} | {s['algorithm']:<10} | "
          f"{pcount:>7,} | {dist_km:>9.2f} | {dur_min:>5.1f} | {r['status'][:30]}")
    
    time.sleep(0.3)

total_elapsed = time.time() - t_start
ok_count = sum(1 for s in scenarios if s.get('polyline_status') == 'OK')
print(f"\n[전체 소요] {total_elapsed:.1f}초")
print(f"[성공] {ok_count}/{len(scenarios)} 시나리오")

[Naver Directions 5 호출 시작 — 18 시나리오]

idx  | 군구         | algo       |   path점 |    거리(km) |     분 | status
---------------------------------------------------------------------------
1    | ganghwa    | Greedy     |     373 |      9.46 | 114.7 | OK
2    | ganghwa    | BruteForce |     373 |      9.46 | 114.7 | OK
3    | junggu     | Greedy     |     267 |      4.84 |  66.7 | OK
4    | junggu     | BruteForce |     209 |      3.84 |  51.1 | OK
5    | donggu     | Greedy     |     275 |      4.00 |  53.8 | OK
6    | donggu     | BruteForce |     173 |      3.71 |  48.3 | OK
7    | michuhol   | Greedy     |     182 |      5.26 |  67.0 | OK
8    | michuhol   | BruteForce |     182 |      5.26 |  67.0 | OK
9    | seogu      | Greedy     |     210 |      4.14 |  54.8 | OK
10   | seogu      | BruteForce |     210 |      4.14 |  54.8 | OK
11   | yeonsu     | Greedy     |     255 |      5.86 |  75.8 | OK
12   | yeonsu     | BruteForce |     246 |      5.70 |  73.9 | OK
13   | namdong    | Gree

In [25]:
# 셀 23 (5-6 (3)): polyline ↔ 회피 영역 shapely 교차 검사

from shapely.geometry import LineString

def line_length_haversine_m(geom):
    """LineString/MultiLineString의 총 길이를 미터로 (Haversine 합산)"""
    if geom.is_empty:
        return 0.0
    total = 0.0
    if geom.geom_type == 'LineString':
        coords = list(geom.coords)
        for j in range(len(coords) - 1):
            total += haversine_km(coords[j][1], coords[j][0],
                                   coords[j+1][1], coords[j+1][0]) * 1000
    elif geom.geom_type in ('MultiLineString', 'GeometryCollection'):
        for sub in geom.geoms:
            total += line_length_haversine_m(sub)
    return total

print(f"[옵션 Y 검증 — polyline ↔ 회피 영역 교차 검사]\n")
print(f"{'idx':<4} | {'군구':<10} | {'algo':<10} | {'교차':<4} | "
      f"{'교차길이(m)':>11} | {'비율':>6} | 교차 UQA")
print("-" * 80)

for i, s in enumerate(scenarios):
    if s.get('polyline_status') != 'OK' or not s.get('polyline'):
        s.update({'crosses_avoid': False, 'intersection_length_m': 0,
                  'intersection_ratio': 0.0, 'crossed_uqa_codes': []})
        print(f"{i+1:<4} | {s['district']:<10} | {s['algorithm']:<10} | -    | "
              f"{'-':>11} | {'-':>6} | (polyline 없음)")
        continue
    
    # polyline (lat, lon) → shapely (x=lon, y=lat) 순서
    line = LineString([(pt[1], pt[0]) for pt in s['polyline']])
    crosses = avoid_union_prepared.intersects(line)
    s['crosses_avoid'] = crosses
    
    if not crosses:
        s['intersection_length_m'] = 0
        s['intersection_ratio'] = 0.0
        s['crossed_uqa_codes'] = []
        print(f"{i+1:<4} | {s['district']:<10} | {s['algorithm']:<10} | No   | "
              f"{'0':>11} | {'0.0%':>6} | -")
        continue
    
    # 교차 구간 추출 및 길이 계산
    intersection = line.intersection(avoid_union)
    inter_m = line_length_haversine_m(intersection)
    line_m = line_length_haversine_m(line)
    ratio = inter_m / line_m if line_m > 0 else 0
    
    # 교차한 UQA 코드 식별
    crossed_uqa = set()
    for code, union in avoid_union_by_code.items():
        if line.intersects(union):
            crossed_uqa.add(code)
    
    s['intersection_length_m'] = inter_m
    s['intersection_ratio'] = ratio
    s['crossed_uqa_codes'] = sorted(crossed_uqa)
    
    print(f"{i+1:<4} | {s['district']:<10} | {s['algorithm']:<10} | YES  | "
          f"{inter_m:>11.1f} | {ratio*100:>5.2f}% | {','.join(sorted(crossed_uqa))}")

# 집계
crossed_count = sum(1 for s in scenarios if s.get('crosses_avoid'))
print(f"\n{'=' * 60}")
print(f"[옵션 Y 검증 집계]")
print(f"{'=' * 60}")
print(f"  교차 시나리오: {crossed_count}/{len(scenarios)}개")

if crossed_count > 0:
    # 알고리즘별
    print(f"\n[알고리즘별 polyline 교차]")
    for algo in ['Greedy', 'BruteForce']:
        algo_s = [s for s in scenarios if s['algorithm'] == algo]
        algo_crossed = [s for s in algo_s if s.get('crosses_avoid')]
        if algo_crossed:
            avg_len = sum(s['intersection_length_m'] for s in algo_crossed) / len(algo_crossed)
            avg_ratio = sum(s['intersection_ratio'] for s in algo_crossed) / len(algo_crossed)
            print(f"  {algo:<11}: 교차 {len(algo_crossed)}/{len(algo_s)}, "
                  f"평균 교차 {avg_len:.1f}m ({avg_ratio*100:.2f}%)")
        else:
            print(f"  {algo:<11}: 교차 0/{len(algo_s)}")
    
    # 군구별 차이 분석
    print(f"\n[군구별 polyline 교차 차이]")
    districts_with_cross = set(s['district'] for s in scenarios if s.get('crosses_avoid'))
    for district in sorted(districts_with_cross):
        g = next((s for s in scenarios if s['district'] == district and s['algorithm'] == 'Greedy'), None)
        b = next((s for s in scenarios if s['district'] == district and s['algorithm'] == 'BruteForce'), None)
        g_str = f"{g['intersection_length_m']:.0f}m ({g['intersection_ratio']*100:.1f}%)" if g and g.get('crosses_avoid') else "교차 없음"
        b_str = f"{b['intersection_length_m']:.0f}m ({b['intersection_ratio']*100:.1f}%)" if b and b.get('crosses_avoid') else "교차 없음"
        print(f"  {district:<10}: Greedy {g_str:<20} | BF {b_str}")
else:
    print(f"\n→ 18개 시나리오의 polyline 모두 회피 영역 외부.")
    print(f"  옵션 Y 검증 결론: 9 OD 표본에서 polyline도 회피 영역 통과 사례 없음.")
    print(f"  단계 6에서 일반화 검증 필요.")

[옵션 Y 검증 — polyline ↔ 회피 영역 교차 검사]

idx  | 군구         | algo       | 교차   |     교차길이(m) |     비율 | 교차 UQA
--------------------------------------------------------------------------------
1    | ganghwa    | Greedy     | No   |           0 |   0.0% | -
2    | ganghwa    | BruteForce | No   |           0 |   0.0% | -
3    | junggu     | Greedy     | YES  |      1032.6 | 21.35% | UQA320,UQA330
4    | junggu     | BruteForce | YES  |      1032.6 | 26.87% | UQA320,UQA330
5    | donggu     | Greedy     | YES  |       171.0 |  4.28% | UQA320
6    | donggu     | BruteForce | YES  |      1782.3 | 48.01% | UQA320
7    | michuhol   | Greedy     | No   |           0 |   0.0% | -
8    | michuhol   | BruteForce | No   |           0 |   0.0% | -
9    | seogu      | Greedy     | No   |           0 |   0.0% | -
10   | seogu      | BruteForce | No   |           0 |   0.0% | -
11   | yeonsu     | Greedy     | No   |           0 |   0.0% | -
12   | yeonsu     | BruteForce | No   |           0 |   0.0% | -

In [ ]:
# 셀 24 (5-7): 단계 5 결과 종합 + folium 시각화 (Tmap 도보 polyline 기준)
import folium

# =========================================================================
# 1. 결과 종합 표 출력 (scenarios에서 자동 계산)
# =========================================================================
print(f"\n{'=' * 70}")
print(f"[단계 5 결과 종합 — Tmap 도보 polyline 기준]")
print(f"{'=' * 70}")

# 옵션 X 검증 — 자동 계산
total_hotspots = len(df_hotspots)
hotspot_in_avoid = df_hotspots['in_avoid_zone'].sum()
total_waypoints_count = len(df_waypoints)
waypoint_in_avoid = df_waypoints['in_avoid_zone'].sum()

print(f"\n[옵션 X 검증 — 경유지 회피]")
print(f"  데이터셋 회피 핫스팟: {hotspot_in_avoid}/{total_hotspots:,} "
      f"({hotspot_in_avoid/total_hotspots*100:.2f}%)")
print(f"  9 OD 경유지 선정 회피: {waypoint_in_avoid}/{total_waypoints_count} "
      f"({waypoint_in_avoid/total_waypoints_count*100:.1f}%)")
# 셀 24 상단의 옵션 X 결론 부분만 교체
if waypoint_in_avoid == 0:
    print(f"  결론: 9 OD 표본에서 회피 핫스팟 선정 0건 관찰.")
    print(f"        단, OD가 도심 시설 위주라 Buffer 안에 회피 핫스팟이 적었을 가능성 (표본 편향).")
    print(f"        단계 6-α의 worst case OD로 가설 검증 필요")

# 옵션 Y 검증 — 자동 계산
crossed_scenarios = [s for s in scenarios if s.get('crosses_avoid')]
crossed_count = len(crossed_scenarios)

print(f"\n[옵션 Y 검증 — Tmap 도보 polyline]")
print(f"  18 시나리오 중 교차: {crossed_count}/{len(scenarios)} "
      f"({crossed_count/len(scenarios)*100:.0f}%)")

for algo in ['Greedy', 'BruteForce']:
    algo_all = [s for s in scenarios if s['algorithm'] == algo]
    algo_crossed = [s for s in algo_all if s.get('crosses_avoid')]
    if algo_crossed:
        avg_m = sum(s['intersection_length_m'] for s in algo_crossed) / len(algo_crossed)
        avg_r = sum(s['intersection_ratio'] for s in algo_crossed) / len(algo_crossed)
        print(f"  {algo} 교차: {len(algo_crossed)}/{len(algo_all)} "
              f"(평균 {avg_m:.0f}m, {avg_r*100:.1f}%)")
    else:
        print(f"  {algo} 교차: 0/{len(algo_all)}")

# 군구별 hard case 식별
print(f"\n[Hard Case — OD 양쪽이 회피 영역인 군구]")
hard_districts = set(s['district'] for s in crossed_scenarios)
for d in sorted(hard_districts):
    g = next((s for s in crossed_scenarios if s['district'] == d and s['algorithm'] == 'Greedy'), None)
    b = next((s for s in crossed_scenarios if s['district'] == d and s['algorithm'] == 'BruteForce'), None)
    g_str = f"{g['intersection_length_m']:.0f}m ({g['intersection_ratio']*100:.1f}%)" if g else "없음"
    b_str = f"{b['intersection_length_m']:.0f}m ({b['intersection_ratio']*100:.1f}%)" if b else "없음"
    print(f"  {d:<10}: Greedy {g_str:<22} | BF {b_str}")

print(f"\n[참고 — Naver 차량 polyline 결과 (이전 검증, 보존용)]")
print(f"  차량 기준: 6/18 (33%) — 도보 기준: {crossed_count}/{len(scenarios)} ({crossed_count/len(scenarios)*100:.0f}%)")
print(f"  차이: 차량 polyline이 도보보다 약 1.5배 과대추정 (가장자리 스침 사례 포함)")
print(f"  공통: junggu·donggu hard case는 도보 기준에서도 통과 잔존")

print(f"\n[결론]")
print(f"  옵션 X 추가 가치: 9 OD 표본에서 미관측. 단계 6에서 일반화 검증")
print(f"  옵션 Y 도입 필요성: 입증 (도보 기준 22%, hard case 명확)")
print(f"  4번째 알고리즘 후보 E (Greedy swap) 설계 진입 정당화됨")

# =========================================================================
# 2. folium 지도 생성 (Tmap 도보 polyline 시각화)
# =========================================================================
INCHEON_CENTER = [37.4563, 126.7052]
m = folium.Map(location=INCHEON_CENTER, zoom_start=11, tiles='OpenStreetMap')

# 회피 영역 폴리곤 (배경)
COLORS_AVOID = {"UQA310": "#8B0000", "UQA320": "#E74C3C", "UQA330": "#F39C12"}
avoid_layer = folium.FeatureGroup(name="회피 영역", show=True)
for p in avoid_polygons:
    color = COLORS_AVOID[p['uqa_code']]
    folium.GeoJson(
        p['geom'].__geo_interface__,
        style_function=lambda x, c=color: {
            'color': c, 'weight': 1, 'fillColor': c, 'fillOpacity': 0.4,
        },
        tooltip=f"{p['uqa_code']} · {p['sigg_name']} · {p['uname']}",
    ).add_to(avoid_layer)
avoid_layer.add_to(m)

# polyline — Greedy / BruteForce 별 레이어 분리
for algo, base_color in [('Greedy', '#1976D2'), ('BruteForce', '#388E3C')]:
    algo_layer = folium.FeatureGroup(name=f"{algo} 도보 polyline", show=True)
    for s in scenarios:
        if s['algorithm'] != algo or not s.get('polyline'):
            continue
        
        crosses = s.get('crosses_avoid', False)
        weight = 4 if crosses else 2
        opacity = 0.85 if crosses else 0.6
        
        popup_html = (
            f"<b>{s['district']} · {algo}</b><br>"
            f"도보 거리: {s['polyline_distance_m']/1000:.2f} km<br>"
            f"도보 소요: {s['polyline_duration_ms']/60000:.1f} 분<br>"
            f"회피 교차: {'YES' if crosses else 'No'}<br>"
        )
        if crosses:
            popup_html += (
                f"교차 길이: {s['intersection_length_m']:.1f} m<br>"
                f"교차 비율: {s['intersection_ratio']*100:.2f}%<br>"
                f"교차 UQA: {','.join(s['crossed_uqa_codes'])}"
            )
        
        folium.PolyLine(
            locations=s['polyline'],
            color=base_color,
            weight=weight,
            opacity=opacity,
            popup=folium.Popup(popup_html, max_width=280),
            tooltip=f"{s['district']} · {algo}" + (" [교차]" if crosses else ""),
        ).add_to(algo_layer)
        
        # O, D 마커
        folium.CircleMarker(
            location=s['O'], radius=4, color=base_color, fill=True,
            fillColor='white', fillOpacity=1, tooltip=f"O · {s['district']}",
        ).add_to(algo_layer)
        folium.CircleMarker(
            location=s['D'], radius=4, color=base_color, fill=True,
            fillColor=base_color, fillOpacity=1, tooltip=f"D · {s['district']}",
        ).add_to(algo_layer)
    
    algo_layer.add_to(m)

# 교차 구간 강조 (붉은 두꺼운 선)
intersect_layer = folium.FeatureGroup(name="교차 구간 강조", show=True)
for s in scenarios:
    if not s.get('crosses_avoid') or not s.get('polyline'):
        continue
    line = LineString([(pt[1], pt[0]) for pt in s['polyline']])
    intersection = line.intersection(avoid_union)
    
    if intersection.geom_type == 'LineString':
        sub_lines = [intersection]
    elif intersection.geom_type == 'MultiLineString':
        sub_lines = list(intersection.geoms)
    elif intersection.geom_type == 'GeometryCollection':
        sub_lines = [g for g in intersection.geoms if g.geom_type in ('LineString', 'MultiLineString')]
    else:
        sub_lines = []
    
    for sub in sub_lines:
        if sub.geom_type == 'MultiLineString':
            for sub2 in sub.geoms:
                coords_latlon = [(c[1], c[0]) for c in sub2.coords]
                folium.PolyLine(
                    coords_latlon, color='#D32F2F', weight=6, opacity=0.95,
                    tooltip=f"{s['district']}·{s['algorithm']} 교차 {s['intersection_length_m']:.0f}m"
                ).add_to(intersect_layer)
        else:
            coords_latlon = [(c[1], c[0]) for c in sub.coords]
            folium.PolyLine(
                coords_latlon, color='#D32F2F', weight=6, opacity=0.95,
                tooltip=f"{s['district']}·{s['algorithm']} 교차 {s['intersection_length_m']:.0f}m"
            ).add_to(intersect_layer)

intersect_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# 범례 (Tmap 도보 명시)
legend_html = '''
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 1000;
            background: white; padding: 12px 14px; border: 2px solid #555;
            border-radius: 4px; font-size: 12px;
            font-family: 'Malgun Gothic', sans-serif;
            box-shadow: 2px 2px 6px rgba(0,0,0,0.2);">
  <b>단계 5-7 시각화 (Tmap 도보 기준)</b><br>
  <span style="display:inline-block; width:18px; height:3px; background:#1976D2; vertical-align: middle;"></span> Greedy 도보 polyline (굵게=교차)<br>
  <span style="display:inline-block; width:18px; height:3px; background:#388E3C; vertical-align: middle;"></span> BruteForce 도보 polyline (굵게=교차)<br>
  <span style="display:inline-block; width:18px; height:5px; background:#D32F2F; vertical-align: middle;"></span> 회피 영역 교차 구간<br>
  <span style="display:inline-block; width:14px; height:10px; background:#8B0000; opacity:0.4;"></span> UQA310 전용공업<br>
  <span style="display:inline-block; width:14px; height:10px; background:#E74C3C; opacity:0.4;"></span> UQA320 일반공업<br>
  <span style="display:inline-block; width:14px; height:10px; background:#F39C12; opacity:0.4;"></span> UQA330 준공업
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# HTML 저장
today_str_local = datetime.now().strftime("%y%m%d")
viz_path = OUTPUT_DIR / f"avoidZoneTest_step5_viz_tmap_{today_str_local}.html"
m.save(str(viz_path))
print(f"\n[시각화 저장] {viz_path.resolve()}")
print(f"  크기: {viz_path.stat().st_size / 1024:.1f} KB")
print(f"  (Naver 차량 기준 시각화는 기존 파일에 보존됨)")

m


[단계 5 결과 종합 — Tmap 도보 polyline 기준]

[옵션 X 검증 — 경유지 회피]
  데이터셋 회피 핫스팟: 58/1,535 (3.78%)
  9 OD 경유지 선정 회피: 0/54 (0.0%)
  결론: Greedy/BF가 S_i 기준으로 회피 핫스팟을 자연 배제. 단계 6 일반화 검증 필요

[옵션 Y 검증 — Tmap 도보 polyline]
  18 시나리오 중 교차: 4/18 (22%)
  Greedy 교차: 2/9 (평균 602m, 12.8%)
  BruteForce 교차: 2/9 (평균 1407m, 37.4%)

[Hard Case — OD 양쪽이 회피 영역인 군구]
  donggu    : Greedy 171m (4.3%)            | BF 1782m (48.0%)
  junggu    : Greedy 1033m (21.3%)          | BF 1033m (26.9%)

[참고 — Naver 차량 polyline 결과 (이전 검증, 보존용)]
  차량 기준: 6/18 (33%) — 도보 기준: 4/18 (22%)
  차이: 차량 polyline이 도보보다 약 1.5배 과대추정 (가장자리 스침 사례 포함)
  공통: junggu·donggu hard case는 도보 기준에서도 통과 잔존

[결론]
  옵션 X 추가 가치: 9 OD 표본에서 미관측. 단계 6에서 일반화 검증
  옵션 Y 도입 필요성: 입증 (도보 기준 22%, hard case 명확)
  4번째 알고리즘 후보 E (Greedy swap) 설계 진입 정당화됨

[시각화 저장] D:\yyeong\ploggo\algorithm-test\output\avoidZoneTest_step5_viz_tmap_260605.html
  크기: 1080.8 KB
  (Naver 차량 기준 시각화는 기존 파일에 보존됨)


결론: 9 OD 표본에서 회피 핫스팟 선정 0건 관찰. 단, OD가 도심 시설 위주라 
      Buffer 안에 회피 핫스팟이 적었을 가능성 (표본 편향). 단계 6-α의 
      worst case OD로 가설 검증 필요